In [ ]:
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math
from collections import namedtuple

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets

import matplotlib.pyplot as plt
import plotly.express as px

# interactive panels
import panel as pn
pn.extension('plotly');

import numpy as np
import pandas as pd
import addict

import cv2
import skimage
import sklearn
import scipy.signal

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading

In [ ]:
%matplotlib inline

In [ ]:
# Magics to autoreload submodules when they are modified
%load_ext autoreload
%autoreload 2

In [ ]:
#%% Load OCT study information
#folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')
# this is also folder name

# strip 14
#study_name = 'GHL_pyapp_20250326T1416'
#study_name = 'GHL_pyapp_20250327T1450'


In [ ]:
study_name = 'GHL_pyapp_20250326T1416';

In [ ]:

octstudy = naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder(study_name,folder_octexport_root);

In [ ]:
octstudy.num_oct_files

In [ ]:
octstudy

# Load all OCT files

In [ ]:
octstudy.load_all_octs(make_pv_volume=True);    # we will use the pyvista volumes a bit later for debuggin

In [ ]:
# export individual vtk files
if False:
    for count,octdata in enumerate(octstudy.octdatalist):
        octdata.pvvol.save(octstudy.folder_study_processed/'{:s}_saveasvtk_{:04d}.vtk'.format(octstudy.study_name,count));

# Stack And Merge All OCT Volumes To A VTK using PYVISTA

In [ ]:
mergedvol = octstudy.pv_stack_all_volumes_along_dimension(dimension=1);

In [ ]:
print(mergedvol)

In [ ]:
# write out stacked .vtk file for opening in slicer
if False:
    octstudy.folder_study_processed.mkdir(exist_ok=True);
    fname = octstudy.folder_study_processed/'{:s}_STACKED.vtk'.format(octstudy.name);
    if(fname.exists()):
        print('Stacked volume .vtk file already exists, will not recreate.');
    else:
        mergedvol.save(fname);

# Prepare To Extract Useful Information

In [ ]:
data_extracted = addict.Addict();

# Processing Step 0 - Replace VTK scalars with scaled bytes

In [ ]:
# make vedo volume
vdvol = vedo.Volume(mergedvol);

In [ ]:
# restrict range and re-scale, cast to integer (will reduce memory by 4x)
oct_scalar_min = 30;
oct_scalar_max = 60;
scalars_rescaled_as_int = skimage.util.img_as_ubyte( (np.clip(vdvol.dataset.active_scalars,a_min=oct_scalar_min,a_max=oct_scalar_max)-oct_scalar_min)/(oct_scalar_max-oct_scalar_min) )

vdvol.dataset['OCTintensity'] = scalars_rescaled_as_int;

In [ ]:
# save this byte-adjusted volume
if False:
    fname = octstudy.folder_study_processed/'{:s}_STACKED_RESCALED_{:}to{:}_uint8.vtk'.format(octstudy.name,oct_scalar_min,oct_scalar_max);
    #vdvol.dataset.save()
    if(fname.exists()):
        print('Stacked volume byte-size .vtk file already exists, will not recreate.');
    else:
        mergedvol.save(fname);

# Processing Step X - Cast In-Memory Views To SimpleITK Types

In [ ]:
# Get ITK image without requiring new memory
import itk
import vtk

# create itk_image
itk_image = itk.image_from_vtk_image(vdvol.dataset)

# itk_image
print(itk_image)

In [ ]:
itk_image.GetImageDimension()

In [ ]:
# make simpleitk image
def itkToSimpleITK(itk_image):
    new_sitk_image = sitk.GetImageFromArray(itk.GetArrayViewFromImage(itk_image),isVector=itk_image.GetNumberOfComponentsPerPixel()>1);
    new_sitk_image.SetOrigin(tuple(itk_image.GetOrigin()))
    new_sitk_image.SetSpacing(tuple(itk_image.GetSpacing()))
    new_sitk_image.SetDirection(itk.GetArrayFromMatrix(itk_image.GetDirection()).flatten()) 
    return new_sitk_image;
simgmerged = itkToSimpleITK(itk_image);

# Processing Step 1 - Automatic Find Bounds Of Strip

In [ ]:
size = simgmerged.GetSize()
print(simgmerged.GetSize())

In [ ]:
# Do Binary Threhsolding On Entire Volume Image
step1_binarythresholding_lo = 250;
step1_binarythresholding_hi = 255;
seg = sitk.BinaryThreshold(
    simgmerged, lowerThreshold=step1_binarythresholding_lo, upperThreshold=step1_binarythresholding_hi, insideValue=1, outsideValue=0
)


In [ ]:
# Get numpy array view from simpleitk image (shares memory)
print(seg.GetSize())
nda = sitk.GetArrayViewFromImage(seg);
print(nda.shape)
# note the dimensions permute

In [ ]:
# Show 2d images of summation along each dimension
ndasum0 = nda.sum(axis=0)
ndasum1 = nda.sum(axis=1)
ndasum2 = nda.sum(axis=2)

fig = plt.figure(figsize=(8,6),constrained_layout=True);
fig.suptitle(
"""{:s} --> range={:s}dB to uint8
Analysis of threshold_segmentation={:s} and has image size={:s}px
Views Of Summation Along Various Dimensions""" \
    .format(
        octstudy.name,
        str((oct_scalar_min,oct_scalar_max)),
        str((step1_binarythresholding_lo,step1_binarythresholding_hi)),
        str(seg.GetSize())
    )
)
gs = fig.add_gridspec(2,3);

fig_ax1 = fig.add_subplot(gs[0,0:2]);
fig_ax1.set_title('Sum Along Axis 0',y=0.95,verticalalignment='top')
fig_ax1.imshow(ndasum2,aspect='auto');
fig_ax1.set_xlabel('px');
fig_ax1.set_ylabel('px');

fig_ax2 = fig.add_subplot(gs[:,2]);
fig_ax2.set_title('Sum Along Axis 1',y=0.95,verticalalignment='top')
fig_ax2.imshow(ndasum1,aspect='auto');
fig_ax2.set_xlabel('px');
fig_ax2.set_ylabel('px');

fig_ax3 = fig.add_subplot(gs[1,0:2]);
fig_ax3.set_title('Sum Along Axis 2',y=0.95,verticalalignment='top')
fig_ax3.imshow(ndasum0.T,aspect='auto');
fig_ax3.set_xlabel('px');
fig_ax3.set_ylabel('px');

fig.savefig(octstudy.folder_study_processed/'processing_step1a_{:s}.jpg'.format(time.strftime('%Y%m%d')))

In [ ]:
# Do Automatic Analysis Using Line Down the Middle Of Strip
fig,ax = plt.subplots(3,1,figsize=(12,10),squeeze=False);
fig.suptitle(
"""{:s} --> range={:s}dB to uint8
Analysis of threshold_segmentation={:s} of wi size={:s}px
Top-Down Summation Into Depth, Find Strip Boundaries""" \
    .format(
        octstudy.name,
        str((oct_scalar_min,oct_scalar_max)),
        str((step1_binarythresholding_lo,step1_binarythresholding_hi)),
        str(seg.GetSize())
    )
)

ndasum2 = nda.sum(axis=2);
ax[0,0].imshow(ndasum2,aspect='auto');
ax[0,0].axhline(ndasum2.shape[0]//2,color='white',linestyle='dashed')
ax[0,0].set_xlabel('px')
ax[0,0].set_ylabel('px')

ax[1,0].set_title('Profile of White Dashed Line',y=0.95,verticalalignment='top')
a_whiteline = ndasum2[ndasum2.shape[0]//2,:].copy();
ax[1,0].plot(a_whiteline);
ax[1,0].set_xlabel('px')
ax[1,0].set_ylabel('intensity')

# bin histogram along white line, into 3 bins, with the median being the upper range
median = np.median(a_whiteline)
ax[2,0].set_title('Histrogram of Above with 3 bins, from 0 to median={:}'.format(median),y=0.95,verticalalignment='top')
histret,histedges,_ = ax[2,0].hist(a_whiteline,bins=3,range=(0,median))
ax[2,0].set_xlabel('bin')
ax[2,0].set_ylabel('count of intensity occurrances')

# take the value in middle of 2nd bin
threshold_value = (histedges[2]+histedges[1])/2;

# look for sharp edges transitioning where these bins occur
a_whiteline[a_whiteline<threshold_value] = 0
a_whiteline[a_whiteline>=threshold_value] = 1
transition_edges = np.where(a_whiteline[:-1] != a_whiteline[1:])[0]
# for right edge, start at middle and find right-most
edge_right = transition_edges[transition_edges>(ndasum2.shape[1]//2)][0];
edge_left = transition_edges[transition_edges<=(ndasum2.shape[1]//2)][-1];

ax[0,0].axvline(edge_left,color='yellow',linestyle='dashed')
ax[0,0].axvline(edge_right,color='orange',linestyle='dashed')
ax[1,0].axvline(edge_left,color='yellow',linestyle='dashed')
ax[1,0].axvline(edge_right,color='orange',linestyle='dashed')

# store useful information
data_extracted['auto_bounds']['topdown_edge_left_px'] = edge_left.item();
data_extracted['auto_bounds']['topdown_edge_right_px'] = edge_right.item();

# save figure
fig.savefig(octstudy.folder_study_processed/'processing_step1b_{:s}.jpg'.format(time.strftime('%Y%m%d')))

In [ ]:
# Do automatic analysis using top-down image processing methods

fig,ax = plt.subplots(3,3,figsize=(14,6),squeeze=False);

fig.suptitle(
"""{:s} --> range={:s}dB to uint8
Analysis of threshold_segmentation={:s} of size={:s}px
Automatic analysis using top-down image processing methods, Find Strip Boundaries""" \
    .format(
        octstudy.name,
        str((oct_scalar_min,oct_scalar_max)),
        str((step1_binarythresholding_lo,step1_binarythresholding_hi)),
        str(seg.GetSize())
    )
)

#image = ndasum2[:,edge_left:edge_right];
image = ndasum2

ax[0,0].set_title('Original',y=0.95,verticalalignment='top')
ax[0,0].imshow(image,aspect='auto')

ax[0,1].set_title('Clipped 0-Median',y=0.95,verticalalignment='top')
image2 = np.clip(image,a_min=0,a_max=median)
ax[0,1].imshow(image2,aspect='auto')

ax[1,0].set_title('Scharr Edge filter',y=0.95,verticalalignment='top')
edge_scharr = skimage.filters.scharr(image2);
ax[1,0].imshow(edge_scharr,aspect='auto')

ax[1,1].set_title('Roberts Edge filter',y=0.95,verticalalignment='top')
edge_roberts = skimage.filters.roberts(image2);
ax[1,1].imshow(edge_roberts,aspect='auto')

ax[2,0].set_title('Sobel Edge filter',y=0.95,verticalalignment='top')
edge_sobel = skimage.filters.sobel(image2);
ax[2,0].imshow(edge_sobel,aspect='auto')

ax[2,1].set_title('absSobelV Edge filter',y=0.95,verticalalignment='top')
edge_sobelv = skimage.filters.sobel_v(image2);
ax[2,1].imshow(abs(edge_sobelv),aspect='auto')

ax[0,2].set_title(r'Canny Edge $\sigma$=20',y=0.95,verticalalignment='top')
edge_canny = skimage.feature.canny(image2,sigma=20,low_threshold=0.45,high_threshold=0.55)
ax[0,2].imshow(edge_canny.view(dtype=np.uint8),aspect='auto')


# contours = skimage.measure.find_contours(edge_canny.view(dtype=np.uint8),level=0.5)
# for contour in contours:
#     ax[1,2].plot(contour[:, 1], contour[:, 0], linewidth=2)
# ax[1,2].yaxis.set_inverted(True)

# -- find some boundaries on topdown view using image processing methods
# houghline finding
ax[1,2].set_title(r'CannyBlurred & Hough Lines',y=0.95,verticalalignment='top')
img_blured_edges = cv2.blur(edge_canny.view(dtype=np.uint8)*255,(3,3));
ax[1,2].imshow(img_blured_edges,aspect='auto')

# Straight-line Hough transform
# Set a precision of Scan a 10 degree angle off the normal (this seems to be 90 degrees with this skimage implementation for horizontal-like lines) with resolution of 0.05 degrees.
tested_angles = np.linspace(math.radians(90-10),math.radians(90+10),int( (math.radians(10)*2)/math.radians(0.05) ), endpoint=True)
h, theta, d = skimage.transform.hough_line(img_blured_edges, theta=tested_angles)
hough_line_data = skimage.transform.hough_line_peaks(h, theta, d);
for _, angle, dist in zip(*hough_line_data):
    (x0, y0) = dist * np.array([np.cos(angle), np.sin(angle)])
    ax[1,2].axline((x0, y0), slope=np.tan(angle + np.pi / 2), color='red',linestyle='dotted')

# Calculate a boundary from sum along strip width
along_strip_sum=ndasum2.sum(axis=0);
along_strip_sum_derivative = scipy.signal.savgol_filter(along_strip_sum,window_length=32,polyorder=3,deriv=1)
edge_topdown_left2 = np.argmax(along_strip_sum_derivative);
edge_topdown_right2 = np.argmin(along_strip_sum_derivative);
ax[2,2].set_title(r'Final = Original+Bounds+Hough',y=0.95,verticalalignment='top')
ax[2,2].imshow(image,aspect='auto');
ax[2,2].axvline(edge_topdown_left2,color='yellow',linestyle='--')
ax[2,2].axvline(edge_topdown_right2,color='orange',linestyle='--')
for _, angle, dist in zip(*hough_line_data):
    (x0, y0) = dist * np.array([np.cos(angle), np.sin(angle)])
    ax[2,2].axline((x0, y0), slope=np.tan(angle + np.pi / 2), color='red',linestyle='dotted')

# Extract relevant data
data_extracted['auto_bounds']['topdown_edge_left_px2'] = edge_topdown_left2.item();
data_extracted['auto_bounds']['topdown_edge_right_px2'] = edge_topdown_right2.item();
data_extracted['auto_topdown_houghline'] = hough_line_data;

# Save to processed folder
fig.savefig(octstudy.folder_study_processed/'processing_step1c_{:s}.jpg'.format(time.strftime('%Y%m%d')))

In [ ]:
# Do automatic analysis using Longitiduinal-Section

ndasum0 = sitk.GetArrayViewFromImage(simgmerged).sum(axis=0)
#ndasum1 = nda.sum(axis=1)
#ndasum2 = nda.sum(axis=2)

fig = plt.figure(figsize=(8,8),constrained_layout=True);
fig.suptitle(
"""{:s} --> range={:s}dB to uint8
Analysis of image (not thresholding) with size={:s}px
Some Bounds-finding using Longitiduinal-Section""" \
    .format(
        octstudy.name,
        str((oct_scalar_min,oct_scalar_max)),
        str(seg.GetSize())
    )
)
gs = fig.add_gridspec(3,1);

# fig_ax1 = fig.add_subplot(gs[0,0:2]);
# fig_ax1.set_title('Sum Along Axis 0',y=0.95,verticalalignment='top')
# fig_ax1.imshow(ndasum2,aspect='auto');
# fig_ax1.set_xlabel('px');
# fig_ax1.set_ylabel('px');

# fig_ax2 = fig.add_subplot(gs[:,2]);
# fig_ax2.set_title('Sum Along Axis 1',y=0.95,verticalalignment='top')
# fig_ax2.imshow(ndasum1,aspect='auto');
# fig_ax2.set_xlabel('px');
# fig_ax2.set_ylabel('px');

fig_ax1 = fig.add_subplot(gs[0,0]);
fig_ax1.set_title('Sum Along Axis 2',y=0.95,verticalalignment='top',color='white');
fig_ax1.imshow(ndasum0.T,aspect='auto');
fig_ax1.set_xlabel('px');
fig_ax1.set_ylabel('px');

nda_sum_curve = ndasum0.sum(axis=0)
pks = scipy.signal.find_peaks(nda_sum_curve,prominence=10000,height=10000)
widths = scipy.signal.peak_widths(nda_sum_curve,pks[0])
localmin = scipy.signal.argrelextrema(nda_sum_curve, np.less)[0]
print(pks)
print(widths)
print(localmin)
tallest_peak = np.argmax(pks[1]['peak_heights'])


# look for top edge
data_extracted['auto_bounds']['thickness_depth_top_px'] = localmin[localmin<pks[0][tallest_peak]][-1].item();


fig_ax2 = fig.add_subplot(gs[1,0]);
fig_ax2.set_title('Sum Along Y-axis Of The Left Figure',y=0.95,verticalalignment='top');
fig_ax2.plot(nda_sum_curve);
fig_ax2.plot(pks[0],nda_sum_curve[pks[0]],marker='o',linestyle='',color='orange');
fig_ax2.plot(localmin,nda_sum_curve[localmin],marker='o',linestyle='',color='red');
fig_ax2.set_yscale('log')



# mark top edge
fig_ax1.axhline( data_extracted['auto_bounds']['thickness_depth_top_px'] ,linestyle='dashed',color='springgreen')
fig_ax2.axvline( data_extracted['auto_bounds']['thickness_depth_top_px'] ,linestyle='dashed',color='springgreen')




savgol_of_sum_surve = scipy.signal.savgol_filter(nda_sum_curve,21,polyorder=3,deriv=2);
pks2 = scipy.signal.find_peaks(savgol_of_sum_surve)
fig_ax3 = fig.add_subplot(gs[2,0]);
fig_ax3.set_title('Sav Gol 2nd Derivative of Above',y=0.95,verticalalignment='top');
fig_ax3.plot(savgol_of_sum_surve);
fig_ax3.plot(pks2[0],savgol_of_sum_surve[pks2[0]],marker='o',linestyle='',color='orange');

# look for bottom edge
data_extracted['auto_bounds']['thickness_depth_bot_px'] = pks2[0][pks2[0] > max(pks[0])][1]
fig_ax1.axhline( data_extracted['auto_bounds']['thickness_depth_bot_px'] ,linestyle='dashed',color='red')
fig_ax3.axvline( data_extracted['auto_bounds']['thickness_depth_bot_px'] ,linestyle='dashed',color='red')

# Save to processed folder
fig.savefig(octstudy.folder_study_processed/'processing_step1d_{:s}.jpg'.format(time.strftime('%Y%m%d')))

In [ ]:
ndasum0.sum(axis=0).shape

In [ ]:
pp.pprint(data_extracted)

# Processing Step 2 - Now Could We Save A Smaller Cropped File? Yes

In [ ]:
print( 'Original',simgmerged.GetSize() )
slicer_0 = slice( data_extracted['auto_bounds']['thickness_depth_top_px']  ,  data_extracted['auto_bounds']['thickness_depth_bot_px']  )
slicer_1 = slice( data_extracted['auto_bounds']['topdown_edge_left_px2']  ,  data_extracted['auto_bounds']['topdown_edge_right_px2']  )
cropped_simple_itk_image = simgmerged[ slicer_0, slicer_1, :]
simgmerged_longcropped = simgmerged[ : , slicer_1, :]
print( 'Cropped Simple ITK Image',cropped_simple_itk_image.GetSize() )
print( 'Long-cropped Simple ITK Image',simgmerged_longcropped.GetSize() )
if False:
    # save this byte-adjusted volume
    fname = octstudy.folder_study_processed/'{:s}_STACKED_CROPPED_RESCALED_{:}to{:}_uint8.vtk'.format(octstudy.name,oct_scalar_min,oct_scalar_max);
    #vdvol.dataset.save()
    if(fname.exists()):
        print('Stacked volume byte-size .vtk file already exists, will not recreate.');
    else:
        #mergedvol.save(fname);
        sitkwriter = sitk.ImageFileWriter();
        sitkwriter.SetFileName(fname.as_posix())
        sitkwriter.Execute(cropped_simple_itk_image)

# Processing Step 3 - Along Strip Analysis A

In [ ]:
# Use the automatically determined boundaries along the strip to restrict the depth
# The automatically determined boundaries are too lenient, and initial cross-sections looks funny
#seg_sliced = seg[:,data_extracted['auto_bounds']['topdown_edge_left_px2']:data_extracted['auto_bounds']['topdown_edge_right_px2'],:]
seg_sliced = seg[:,slicer_1,:];

seg_sliced.GetSize()

In [ ]:
import matplotlib.patches
def process_a_crosssection_sliceA(seg : sitk.Image, slab_thickness_px,slice_position_along_strip_px, mkplot=True):
    slab_thickness = slab_thickness_px;
    sliceinfo = {};



    if(slab_thickness>1):
        seg_subset = seg[:,slice_position_along_strip_px-slab_thickness//2:slice_position_along_strip_px+slab_thickness//2,:]
        img_subset = simgmerged_longcropped[:,slice_position_along_strip_px-slab_thickness//2:slice_position_along_strip_px+slab_thickness//2,:]
    else:
        seg_subset = seg[:,slice_position_along_strip_px:slice_position_along_strip_px+1,:]
        img_subset = simgmerged_longcropped[:,slice_position_along_strip_px:slice_position_along_strip_px+1,:]

    nda = sitk.GetArrayFromImage(seg_subset);
    print(nda.shape)
    nda_mean = np.mean(nda,axis=1);
    nda_mean_sum = np.sum(nda_mean,axis=0);

    orignda = sitk.GetArrayViewFromImage(img_subset);
    orignda_mean = np.mean(orignda,axis=1);




    if mkplot:
        #fig,ax = plt.subplots(3,2,figsize=(20,12));
        fig = plt.figure(figsize=(20,12));
        gs = fig.add_gridspec(4,6);
        fig.suptitle('{:s} --> range={:s}dB to uint8\nprocess_a_crosssection_sliceA code:20250428 size={:s}px\nslice_position_along_strip={:.0f}px and slab_thickness={:.0f}px'.format(octstudy.name,str((oct_scalar_min,oct_scalar_max)),str(seg.GetSize()),slice_position_along_strip_px,slab_thickness_px))


    # SHOW FULL IMAGES
    if mkplot:
        ax = fig.add_subplot(gs[0:2,1:3]);
        ax.set_title('1. Orig SlabMean BThresh {:},{:} {:s}px'.format(step1_binarythresholding_lo,step1_binarythresholding_hi,str(orignda_mean.shape)),y=1.0,verticalalignment='bottom')
        ax.imshow(nda_mean,aspect='auto',cmap='grey');
        ax_gray_img_binarythresh = ax;
    
        ax = fig.add_subplot(gs[0:2,4:6]);
        ax.set_title('0. Orig SlabMean uint8 {:},{:}dB {:s}px'.format(oct_scalar_min,oct_scalar_max,str(orignda_mean.shape)),y=1.0,verticalalignment='bottom')
        ax.imshow(orignda_mean,aspect='auto',cmap='grey');
        ax_gray_img_orig = ax;


    # SHOW STEP 2- SUM OF BINARY THRESHOLDED WHOLE-IMAGE
    if mkplot:
        ax = fig.add_subplot(gs[2,1:3])
        ax.set_title('2. Sum Along Y of #1 of Intensities',y=0.95,verticalalignment='top')
        ax.plot(nda_mean_sum);
    
    pixel_depth_strip_top_sum_threshold = 100;
    pixel_depth_strip_top = np.nonzero(nda_mean_sum>pixel_depth_strip_top_sum_threshold)[0][0];
    pixel_depth_strip_bot = np.nonzero(nda_mean_sum>pixel_depth_strip_top_sum_threshold)[0][-1];

    if mkplot:
        #ax[1,1].set_title('Depth-Mean And X-Sum Vs Y',y=0.95,verticalalignment='top')
        ax.plot(pixel_depth_strip_top, nda_mean_sum[pixel_depth_strip_top] , marker='^', markersize=10,color='orange');
        ax.axvline(pixel_depth_strip_top,linestyle='--',color='orange');
        ax.plot(pixel_depth_strip_bot, nda_mean_sum[pixel_depth_strip_bot] , marker='^', markersize=10,color='yellow');
        ax.axvline(pixel_depth_strip_bot,linestyle='--',color='yellow');
        
        ax_gray_img_binarythresh.axvline(pixel_depth_strip_top,linestyle='--',color='orange');
        ax_gray_img_binarythresh.axvline(pixel_depth_strip_bot,linestyle='--',color='yellow');


    # SHOW STEP 3 - Savgol of #2
    nda_mean_sum_curvature = scipy.signal.savgol_filter(nda_mean_sum,21,polyorder=3,deriv=2);
    pks = scipy.signal.find_peaks(abs(nda_mean_sum_curvature),height=0.1,prominence=0)
    # restrict peaks to be right of the white line
    pksculled=pks[0][ pks[0]> round( (pixel_depth_strip_bot-pixel_depth_strip_top)/2 + pixel_depth_strip_top ) ] # halfway between orange and yellow
    widths = scipy.signal.peak_widths(abs(nda_mean_sum_curvature),pksculled,rel_height=0.5)
    print('pks1',pks)
    print('widths1',widths)
    if mkplot:
        #ax[2,1].plot(np.diff(nda_mean_sum,1));
        ax = fig.add_subplot(gs[3,1:3]);
        ax.set_title('3. abs Savgol 2nd Derivative Of #2',y=0.95,verticalalignment='top')
        ax.plot(abs(nda_mean_sum_curvature));
        ax.plot(pksculled,abs(nda_mean_sum_curvature[pksculled]),'o',linestyle='');
    
    #pixel_depth_wax_center = round( (pixel_depth_strip_bot-pixel_depth_strip_top)/2 + pixel_depth_strip_top )
    pixel_depth_wax_center = pksculled[0];
    if mkplot:
        ax.axvline(pixel_depth_wax_center,linestyle='--',color='black');
        ax_gray_img_binarythresh.axvline(pixel_depth_wax_center,linestyle='--',color='white');


    # SHOW STEP 4 - Inverted ROI1
    if mkplot:
        ax = fig.add_subplot(gs[0:2,0])
        ax.set_title('4. Inverted ROI1',y=1.0,verticalalignment='bottom')
    imgtmp = nda_mean[:,pixel_depth_strip_top:pixel_depth_strip_bot];
    if mkplot:
        ax.imshow(np.max(imgtmp)-imgtmp,aspect='auto');
        # show the white line
        ax.axvline(pixel_depth_wax_center-pixel_depth_strip_top,linestyle='--',color='white');
        ax_thresholded_roi1 = ax;
    
    # SHOW STEP 5 - INVERSE SUM OF DEPTH ALONG WHITE LINE OF #4
    sum_of_depth_intensities = np.sum(imgtmp,axis=1);
    sum_of_depth_intensities = max(sum_of_depth_intensities) - sum_of_depth_intensities;
    if mkplot:
        ax = fig.add_subplot(gs[2,0]);
        ax.set_title('5. Sum Along Y on #4',y=0.95,verticalalignment='top');
        ax.plot(sum_of_depth_intensities)
        #ax.plot( scipy.signal.savgol_filter(sum_of_depth_intensities,21,polyorder=3,deriv=2)*100 );
    #pks = scipy.signal.find_peaks(sum_of_depth_intensities,width=20,prominence=1.2)
    pks = scipy.signal.find_peaks(sum_of_depth_intensities,height=25,prominence=25)
    widths = scipy.signal.peak_widths(sum_of_depth_intensities,pks[0],rel_height=0.5)
    print(pks)
    print(widths)
    if mkplot:
        ax.plot(pks[0],sum_of_depth_intensities[pks[0]],linestyle='none',marker='.',color='blue')
    assert(pks[0].shape[0]>=1)
    # retain widths of the single largest peak
    pkidxlargest = np.argmax(widths[0]);
    pkcenter = int(round( widths[3][pkidxlargest]- widths[2][pkidxlargest] )//2 + widths[2][pkidxlargest]);
    if mkplot:
        #ax_thresholded_roi1.axhline(pkcenter,linestyle='--',color='red');
        ax_gray_img_binarythresh.axhline(pkcenter,linestyle='--',color='red');
    px_wax_transverse_edges = ( widths[2][pkidxlargest].item() , widths[3][pkidxlargest].item() )
    if mkplot:
        #ax_thresholded_roi1.axhline((px_wax_transverse_edges[0],px_wax_transverse_edges[1]),linestyle='--',color='purple')
        ax.axvline(px_wax_transverse_edges[0],linestyle='--',color='purple');
        ax.axvline(px_wax_transverse_edges[1],linestyle='--',color='purple');
        ax_thresholded_roi1.axhline(px_wax_transverse_edges[0],linestyle='--',color='purple');
        ax_thresholded_roi1.axhline(px_wax_transverse_edges[1],linestyle='--',color='purple');
        ax_gray_img_binarythresh.axhline(px_wax_transverse_edges[0],linestyle='--',color='purple');
        ax_gray_img_binarythresh.axhline(px_wax_transverse_edges[1],linestyle='--',color='purple');


    # SHOW STEP 6 - SEED POINT SEARCH, TOP RIGHT IN WAX AREA
    # take cross-section along white dashed line
    # take intensity over dash red line out of plane of strip
    if mkplot:
        ax = fig.add_subplot(gs[3,0]);
        ax.set_title('6. Intensity Red Dashed',y=0.95,verticalalignment='top');
    red_outofplane_intensity = nda_mean[pkcenter,:];
    intensity_purple_bounds = nda_mean.sum(axis=0);
    if mkplot:
        ax.plot(red_outofplane_intensity)
        #ax[2,0].plot( intensity_purple_bounds, color='purple' )

    # specify a seed point
    #pks = scipy.signal.find_peaks(red_outofplane_intensity,distance=30)
    #wax_top_seed_candidate_px = (pks[0][0].item(),pkcenter)
    pks = scipy.signal.find_peaks(red_outofplane_intensity[0:pixel_depth_strip_bot],height=0.59,prominence=0,distance=10)
    # discard peaks left of the white dashed line
    pks2culled=pks[0][ pks[0]< pixel_depth_wax_center ] # PEAKS LESS THAN THE WHITE DASHED LINE
    widths = scipy.signal.peak_widths(red_outofplane_intensity[0:pixel_depth_strip_bot],pks2culled,rel_height=0.5)
    pkidxlargest = np.argmax(widths[0]);
    wax_top_seed_candidate_px = (round((pks[1]['right_bases'][pkidxlargest]-pks[1]['left_bases'][pkidxlargest])/2 + pks[1]['left_bases'][pkidxlargest]) ,pkcenter)
    print('pks2',pks)
    print('wid2',widths)
    if mkplot:
        ax.plot(pks[0],red_outofplane_intensity[pks[0]],linestyle='none',marker='.',color='blue')
        ax_gray_img_binarythresh.plot(*wax_top_seed_candidate_px, 'x', markersize=10 , color='cyan' );
        ax.axvline(wax_top_seed_candidate_px[0],linestyle='--',color='cyan');

    # aggregate Part A1 data
    sliceinfo['pixel_depth_strip_top_sum_threshold'] = pixel_depth_strip_top_sum_threshold;
    sliceinfo['pixel_depth_strip_top'] = pixel_depth_strip_top.item();
    sliceinfo['pixel_depth_strip_bot'] = pixel_depth_strip_bot.item();
    sliceinfo['pixel_depth_wax_center'] = pixel_depth_wax_center;
    sliceinfo['px_wax_transverse_edges'] = px_wax_transverse_edges;
    sliceinfo['wax_top_seed_candidate_px'] = wax_top_seed_candidate_px;



    #----- MOVE ON TO PART 3B STUFF -----
    #slicex = slice(sliceinfo['wax_top_seed_candidate_px'][1]-data_extracted['auto_wax']['max_transverse_wax_edge_width_px']//2 , sliceinfo['wax_top_seed_candidate_px'][1]+data_extracted['auto_wax']['max_transverse_wax_edge_width_px']//2)
    #slicex = slice(sliceinfo['wax_top_seed_candidate_px'][1] , sliceinfo['wax_top_seed_candidate_px'][1]+data_extracted['auto_wax']['max_transverse_wax_edge_width_px'])
    slicex = slice( round(sliceinfo['px_wax_transverse_edges'][0]) , round(sliceinfo['px_wax_transverse_edges'][1]) )
    #slicex = slice( round(sliceinfo['px_wax_transverse_edges'][0])-10 , round(sliceinfo['px_wax_transverse_edges'][1])+10 )

    #slicey = slice(sliceinfo['pixel_depth_strip_top'] , sliceinfo['pixel_depth_strip_top']+3*(sliceinfo['pixel_depth_strip_bot']-sliceinfo['pixel_depth_strip_top'])  )
    #slicey = slice(sliceinfo['pixel_depth_strip_top'] , sliceinfo['pixel_depth_strip_top']+6*(sliceinfo['pixel_depth_strip_bot']-sliceinfo['pixel_depth_strip_top'])  )
    slicey = slice(sliceinfo['pixel_depth_strip_top'] , int(orignda.shape[2]*0.9)  )
    #slicey = slice(sliceinfo['pixel_depth_strip_top'] , orignda.shape[2]  )
    #slicey = slice(sliceinfo['wax_top_seed_candidate_px'][0] , sliceinfo['pixel_depth_strip_top']+3*(sliceinfo['pixel_depth_strip_bot']-sliceinfo['wax_top_seed_candidate_px'][0])  )
    print('slicex',slicex);
    print('slicey',slicey);
    image = nda_mean[  slicex , slicey ];
    image = skimage.util.img_as_ubyte(image);

    imageorig = orignda_mean[  slicex , slicey ];
    imageorig = skimage.util.img_as_ubyte(imageorig/imageorig.max());


    # ADD RED ROI2 RECTANGLES TO BIG IMAGES
    if mkplot:
        # add red roi2 rectangle ro #0 and #1 plots
        rect = matplotlib.patches.Rectangle( (slicey.start, slicex.start), slicey.stop-slicey.start, slicex.stop-slicex.start, linewidth=1, edgecolor='r', facecolor='none')
        ax_gray_img_binarythresh.add_patch(rect)
        rect = matplotlib.patches.Rectangle( (slicey.start, slicex.start), slicey.stop-slicey.start, slicex.stop-slicex.start, linewidth=1, edgecolor='r', facecolor='none')
        ax_gray_img_orig.add_patch(rect);
        #ax_orig_b = ax;

    # SHOW STEP 6a,b - CROP TO ROI2 FOR SLICE
    if mkplot:
        ax = fig.add_subplot(gs[0,3])
        ax.set_title('6a. Crop To ROI2 from #1',y=1.0,verticalalignment='bottom')
        ax.imshow(image,aspect='auto',)
        ax.axvline(sliceinfo['pixel_depth_strip_bot']-sliceinfo['pixel_depth_strip_top'],color='yellow',linestyle='dashed')
        ax.plot(sliceinfo['wax_top_seed_candidate_px'][0]-slicey.start,sliceinfo['wax_top_seed_candidate_px'][1]-slicex.start,marker='x',markersize=10)
        ax_roi2_a = ax;

        ax = fig.add_subplot(gs[2,3])
        ax.set_title('6b. Crop To ROI2 from #0',y=1.0,verticalalignment='bottom')
        ax.imshow(imageorig,aspect='auto',)
        ax.axvline(sliceinfo['pixel_depth_strip_bot']-sliceinfo['pixel_depth_strip_top'],color='yellow',linestyle='dashed')
        ax.plot(sliceinfo['wax_top_seed_candidate_px'][0]-slicey.start,sliceinfo['wax_top_seed_candidate_px'][1]-slicex.start,marker='x',markersize=10)
        ax_roi2_b = ax;


    ## SHOW STEP 7b - denoise
    # if mkplot:
    #     ax = fig.add_subplot(gs[0,3])
    #     ax.set_title('1. Denoise of #0',y=1.0,verticalalignment='bottom')
    # # denoise image
    # denoised = skimage.filters.rank.median(image, skimage.morphology.disk(2))
    # if mkplot:
    #     ax.imshow(denoised,aspect='auto')
    #     #ax.axvline(sliceinfo['pixel_depth_strip_bot']-slicey.start,color='yellow',linestyle='dashed')
    #     ax.plot(sliceinfo['wax_top_seed_candidate_px'][0]-slicey.start,sliceinfo['wax_top_seed_candidate_px'][1]-slicex.start,marker='x',markersize=10)
    #     ax1a = ax;
    
    if mkplot:
        ax = fig.add_subplot(gs[2,4])
        ax.set_title('7b. Denoise of #6b',y=1.0,verticalalignment='bottom')
    # denoise image
    denoisedorig = skimage.filters.rank.median(imageorig, skimage.morphology.disk(2))
    if mkplot:
        ax.imshow(denoisedorig,aspect='auto')
        #ax.axvline(sliceinfo['pixel_depth_strip_bot']-slicey.start,color='yellow',linestyle='dashed')
        ax.plot(sliceinfo['wax_top_seed_candidate_px'][0]-slicey.start,sliceinfo['wax_top_seed_candidate_px'][1]-slicex.start,marker='x',markersize=10)
        #ax1b = ax;
        ax_roi2_denoised_b = ax;


    ## SHOW STEP 8a and 8b - sum along X on #6
    image_sumxB = np.sum(image,axis=0);
    pksA = scipy.signal.find_peaks(image_sumxB,prominence=(100,None),width=(1,200),distance=30);
    sliceinfo['pksA'] = pksA;
    print('sum peaksA',pksA)
    if mkplot:
        ax = fig.add_subplot(gs[1,3])
        ax.set_title('8a. Sum Along X of #6a',y=1.0,verticalalignment='bottom')
        ax.semilogy(image_sumxB)
        ax.semilogy(pksA[0],image_sumxB[pksA[0]],marker='o',linestyle='')
        ax.axvline(sliceinfo['pixel_depth_strip_bot']-slicey.start,color='yellow',linestyle='dashed')
        ax.axvline(sliceinfo['wax_top_seed_candidate_px'][0]-slicey.start,color='cyan',linestyle='dashed')

    image_sumx_origB = np.sum(denoisedorig,axis=0);
    pksB = scipy.signal.find_peaks(image_sumx_origB,prominence=(100,None),width=(1,200),distance=30);
    sliceinfo['pksB'] = pksB;
    #widths = scipy.signal.wi
    print('sum peaksB',pksB)

    # # select peaks between pksA and pksB
    # if((pksB[0].shape[0]-pksA[0].shape[0])<=1):
    #     print('pksA shape and pksB shape differs by <=1')
    #     pksApksB_distance_to_be_same=35;
    #     overlapping_peak_indexA = [np.argmax(abs(pksB[0]-x)<pksApksB_distance_to_be_same).item() for x in pksA[0]];
    #     selected_peak_in_B = overlapping_peak_indexA[-1];
    # elif((pksB[0].shape[0]-pksA[0].shape[0])>1):
    #     print('pksA shape and pksB shape differs by >1')
    #     #pksApksB_distance_to_be_same=35;
    #     #overlapping_peak_indexA = [np.argmax(abs(pksB[0]-x)<pksApksB_distance_to_be_same).item() for x in pksA[0]];
    #     #selected_peak_in_B = overlapping_peak_indexA[-1];
    #     selected_peak_in_B = pksA[0].shape[0];
    # else:
    #     raise NotImplementedError('unhandled')
    
    # sliceinfo['wax_roi_valve_bottom'] = pksB[0][selected_peak_in_B].item();
    if mkplot:
        ax = fig.add_subplot(gs[3,3])
        ax.set_title('8b. Sum Along X of #7b',y=1.0,verticalalignment='bottom')
        ax.semilogy(image_sumx_origB)
        ax.semilogy(pksB[0],image_sumx_origB[pksB[0]],marker='o',linestyle='')
        ax.axvline(sliceinfo['pixel_depth_strip_bot']-slicey.start,color='yellow',linestyle='dashed')
        ax.axvline(sliceinfo['wax_top_seed_candidate_px'][0]-slicey.start,color='cyan',linestyle='dashed')
        #ax.axvline(sliceinfo['wax_roi_valve_bottom'],color='magenta',linestyle='dashed');
        ax_sums_8b = ax;

    image_sumx_origB_detrend = scipy.signal.detrend(image_sumx_origB);
    pksB2 = scipy.signal.find_peaks(image_sumx_origB_detrend,prominence=(100,None),width=(1,200),distance=30,height=20);
    sliceinfo['pksB2'] = pksB2;
    #widths = scipy.signal.wi
    print('sum peaksB2',pksB2)
    if mkplot:
        ax = fig.add_subplot(gs[3,4])
        ax.set_title('10b. Detrend of #8b',y=1.0,verticalalignment='bottom')
        ax.plot(image_sumx_origB_detrend)
        ax.plot(pksB2[0],image_sumx_origB_detrend[pksB2[0]],marker='o',linestyle='')
    
    # # select peaks between pksA and pksB
    # if((pksB[0].shape[0]-pksA[0].shape[0])<=1):
    #     print('pksA shape and pksB shape differs by <=1')
    #     pksApksB_distance_to_be_same=35;
    #     overlapping_peak_indexA = [np.argmax(abs(pksB[0]-x)<pksApksB_distance_to_be_same).item() for x in pksA[0]];
    #     selected_peak_in_B = overlapping_peak_indexA[-1];
    # elif((pksB[0].shape[0]-pksA[0].shape[0])>1):
    #     print('pksA shape and pksB shape differs by >1')
    #     #pksApksB_distance_to_be_same=35;
    #     #overlapping_peak_indexA = [np.argmax(abs(pksB[0]-x)<pksApksB_distance_to_be_same).item() for x in pksA[0]];
    #     #selected_peak_in_B = overlapping_peak_indexA[-1];
    #     selected_peak_in_B = pksA[0].shape[0];
    # else:
    #     raise NotImplementedError('unhandled')
    #selected_peak_in_B = np.argmax(pksB2[1]['prominences']);
    #pksmod = pksB2[0][pksB2[0] > sliceinfo['pixel_depth_strip_bot']-slicey.start];
    #pksmodheights = pksB2[0][pksB2[0] > sliceinfo['pixel_depth_strip_bot']-slicey.start];
    pksqualifying = np.where(pksB2[0] > sliceinfo['pixel_depth_strip_bot']-slicey.start)[0]; # all to the right of the yellow line
    print('pksqualifying',pksqualifying);
    if(pksqualifying.shape[0]>1):
        # selected_peak_in_B = sorted(range(pksB2[0].shape[0]),key= lambda x: pksB2[1]['peak_heights'][x],reverse=True)[1];
        # sliceinfo['wax_roi_valve_bottom'] = pksB2[0][selected_peak_in_B].item();

        #selected_peak_in_B = sorted( range(pksqualifying.shape[0]),key= lambda x: pksB2[1]['peak_heights'][pksqualifying][x],reverse=True)[0];
        selected_peak_in_B = sorted( range(pksqualifying.shape[0]),key= lambda x: pksB2[1]['widths'][pksqualifying][x],reverse=True)[0];
        print(selected_peak_in_B)
        sliceinfo['wax_roi_valve_bottom'] = pksB2[0][pksqualifying][selected_peak_in_B].item();

    else:
        #sliceinfo['wax_roi_valve_bottom'] = pksB2[0][0].item();
        if(pksqualifying.shape[0]==0):
            # use pksB
            print('BOTTOM SEARCH: odd condition, using pksB instead of pksB2')
            pksqualifying = np.where(pksB[0] > sliceinfo['pixel_depth_strip_bot']-slicey.start)[0]; # all to the right of the yellow line
            selected_peak_in_B = sorted( range(pksqualifying.shape[0]),key= lambda x: pksB[1]['widths'][pksqualifying][x],reverse=True)[0];
            sliceinfo['wax_roi_valve_bottom'] = pksB[0][pksqualifying][selected_peak_in_B].item();
        else:
            # pick first peak from pksB2
            sliceinfo['wax_roi_valve_bottom'] = pksB2[0][pksqualifying][0].item();
    
    if mkplot:
        ax.axvline(sliceinfo['wax_roi_valve_bottom'],color='magenta',linestyle='dashed');



    # SHOW BOTTOM OF WAX
    if mkplot:
        ax_sums_8b.axvline(sliceinfo['wax_roi_valve_bottom'],color='magenta',linestyle='dashed');
        ax_roi2_a.axvline(sliceinfo['wax_roi_valve_bottom'],color='magenta',linestyle='dashed');
        #ax_roi2_a.axvline(sliceinfo['wax_roi_valve_bottom'],color='red',linestyle='dashed')
        ax_roi2_b.axvline(sliceinfo['wax_roi_valve_bottom'],color='magenta',linestyle='dashed');
        ax_roi2_denoised_b.axvline(sliceinfo['wax_roi_valve_bottom'],color='magenta',linestyle='dashed');
        #ax_roi2_b.axvline(sliceinfo['wax_roi_valve_bottom'],color='red',linestyle='dashed')


    ## SHOW STEP 9b - OTSU threshold
    ## -- fig3
    # Applying multi-Otsu threshold for the default value, generating
    # several classes.
    thresholds = skimage.filters.threshold_multiotsu(denoisedorig,classes=5)
    # Using the threshold values, we generate the three regions.
    regions = np.digitize(denoisedorig, bins=thresholds)
    #print('5. Multiotsu Thresholds',thresholds);
    if mkplot:
        ax = fig.add_subplot(gs[2,5])
        ax.set_title('9b. Multi-Otsu of #7b\n{:s}'.format(str(thresholds)),y=1.0,verticalalignment='bottom')
        ax.imshow(regions,cmap='jet',aspect='auto')
    sliceinfo['testseg_otsu_regions'] = regions;
    sliceinfo['testseg_otsu_thresholds'] = thresholds;



    if mkplot:
        fig.tight_layout();
    else:
        fig = None;
    



    return (
        # SliceDataCrossSection(
        #     pixel_depth_strip_top_sum_threshold, , pixel_depth_strip_bot.item(), pixel_depth_wax_center, px_wax_transverse_edges,
        #     wax_top_seed_candidate_px
        # ),
        sliceinfo,
        fig if mkplot else None
    )

#process_a_crosssection_slice(seg_sliced,10,seg_sliced.TransformPhysicalPointToIndex( (0,100,0) )[1])
#process_a_crosssection_slice(seg_sliced,10,2875)
#process_a_crosssection_sliceA(seg_sliced,10,1500)
##process_a_crosssection_sliceA(seg_sliced,10,320)
#process_a_crosssection_sliceA(seg_sliced,10,1605)
process_a_crosssection_sliceA(seg_sliced,10,2495)
#process_a_crosssection_sliceA(seg_sliced,10,5055)
#process_a_crosssection_sliceA(seg_sliced,10,5050)
#process_a_crosssection_sliceA(seg_sliced,10,4990)

In [ ]:
# loop and calculate and make figures
slab_thickness = 10;

mkplot=True;
overwrite=True;

seg.GetSize()[1]
if overwrite:
    datalist = [];
#np.linspace(start=slab_thickness//2,stop=Seg.GetSize()[1],num)
slice_centers = list(range(slab_thickness//2, seg_sliced.GetSize()[1], slab_thickness//2))
#slice_centers = list(range(slab_thickness//2, seg_sliced.GetSize()[1], slab_thickness))
import os
filebase = r'figureoutput_20250428_stepA';
for count,slicecenter in enumerate(slice_centers):
    print(count,slicecenter)
    output_filename = r'C:\TEMP\{:s}{:04d}.jpg'.format(filebase,count);

    if( (overwrite and os.path.exists(output_filename)) or (not os.path.exists(output_filename))):
        data,fig = process_a_crosssection_sliceA(seg_sliced,slab_thickness,slicecenter,mkplot=mkplot);
        
        if mkplot:
            fig.savefig(output_filename);
            print('Wrote',output_filename);
            plt.close(fig);
        
        datalist.append(data);
    
    #break;

In [ ]:
# make video from image sequence
import ffmpeg
import glob
import os
try:
    (
    ffmpeg
    .input(r'C:\TEMP\{:s}%04d.jpg'.format(filebase), framerate=30) # Assumes images are named frame_1.png, frame_2.png etc.
    #.output(r'C:\TEMP\{:s}.mp4'.format(filebase), crf=20, preset='slower', movflags='faststart', pix_fmt='yuv420p')
    .output(octstudy.folder_study_processed/r'{:s}.mp4'.format(filebase), crf=20, preset='slower', movflags='faststart', pix_fmt='yuv420p')
    .run(capture_stdout=True, capture_stderr=True, overwrite_output=True)
    )
except ffmpeg.Error as e:
    print('stdout:', e.stdout.decode('utf8'))
    print('stderr:', e.stderr.decode('utf8'))
    raise e
# delete the source .jpgs
for filepath in glob.glob(r'C:\TEMP\{:s}*.jpg'.format(filebase)):
    os.unlink(filepath);


In [ ]:
# make a dfstepA dataframe
#dfstepA = pd.DataFrame.from_records(datalist,columns=datalist[0]._fields);
dfstepA = pd.DataFrame.from_records(datalist);
dfstepA['slice'] = slice_centers;
dfstepA = dfstepA.set_index('slice');
dfstepA.to_hdf(octstudy.folder_study_processed/'along_strip_data_extracted.hdf5',key='dfstepA')

In [ ]:
# extract max transverse edge width
max_transverse_wax_edge_width = np.max(np.diff(np.vstack(dfstepA['px_wax_transverse_edges'].apply(np.array)),axis=1));
data_extracted['auto_wax']['max_transverse_wax_edge_width_px'] = round(max_transverse_wax_edge_width.item());

In [ ]:
# Save Extracted Information To File
np.savez(octstudy.folder_study_processed/'data_extracted.npz',data_extracted=data_extracted.to_dict())

In [ ]:
# Load Previously-Extracted Data From Files
if False:
    data_extracted = addict.Addict( np.load(octstudy.folder_study_processed/'data_extracted.npz',allow_pickle=True)['data_extracted'].item() )
    dfstepA = pd.read_hdf(octstudy.folder_study_processed/'along_strip_data_extracted.hdf5',key='dfstepA')
if False:
    store = pd.HDFStore(octstudy.folder_study_processed/'along_strip_data_extracted.hdf5')

In [ ]:
dfstepA

In [ ]:
#parray = np.vstack(dfstepA['px_wax_transverse_edges'].apply(np.array))
parray = np.vstack(dfstepA['wax_top_seed_candidate_px'].apply(np.array))
parray = np.insert(parray,1,dfstepA.index.values,axis=1);

In [ ]:
fig = px.scatter(parray[:,0])
fig.show()

In [ ]:
pl = pv.Plotter(notebook=False);
pl.background_color='gray'

poly_wax_line_top = pv.PolyData(parray);
pl.add_mesh( poly_wax_line_top )

pl.show()

In [ ]:
poly_wax_line_top.plot(notebook=False)

In [ ]:
tuple(parray[0,:].tolist())

In [ ]:
pl = pv.Plotter(notebook=False);
pl.background_color='gray'

poly_wax_line_top = pv.PolyData( np.apply_along_axis( func1d=lambda x: seg_sliced.TransformIndexToPhysicalPoint(tuple(x.tolist())), arr=parray, axis=1) );
pl.add_mesh( poly_wax_line_top )

pl.show()

In [ ]:
pp.pprint(data_extracted)

# Processing Step 4 - Analyze The Cross-section DataA

In [ ]:
dfstepA.shape[0]

In [ ]:
# process this datalist3

# --INITIALIZATION--
# initialize at the half-way mark down the strip
halfway_datalist_index = dfstepA.shape[0]//2;
halfway_datalist_pxslice = slice_centers[halfway_datalist_index];
print('Initialize At Halfway Index',halfway_datalist_index);
print('Initialize At Halfway Pixel {:}px'.format(halfway_datalist_pxslice));

def guess_waxbottom_from_peaks_close_to_lastvalue(sliceinfo,last_pos=None):

    # 1st tallest peak is the wax valve top
    # assume 2nd tallest peak is the wax valve or strip bottom
    set_last_pos = True;

    pks = sliceinfo['pksB2'];
    pksqualifying = np.where(pks[0] > sliceinfo['pixel_depth_strip_bot']-sliceinfo['pixel_depth_strip_top'])[0]; # all to the right of the yellow line
    print('pksqualifying',pksqualifying);
    if(pksqualifying.shape[0]>1):

        tmp = sorted(list(range(pks[0][pksqualifying].shape[0])),key=lambda x: pks[1]['width_heights'][pksqualifying][x],reverse=True);
        candidate_pkidx = tmp[0]; # Widest peak past the yellow-line
        wax_bottom_depth = pks[0][pksqualifying][candidate_pkidx];

        # # pksqualifying = np.where(pksB2[0] > sliceinfo['pixel_depth_strip_bot']-slicey.start)[0]; # all to the right of the yellow line
        # # print('pksqualifying',pksqualifying);
        # # if(pksqualifying.shape[0]>1):

        # # catch a certain condition where pksB has many more features thans pksB2 and pksA
        # if( (sliceinfo['pksB'][0].shape[0]>sliceinfo['pksB2'][0].shape[0]) and (sliceinfo['pksB'][0].shape[0]>sliceinfo['pksB2'][0].shape[0]) ):
        #     print('condition catch: 1');

        #     # use pksB data only
            
        #     tmp = sorted(list(range(sliceinfo['pksB'][0].shape[0])),key=lambda x: sliceinfo['pksB'][1]['width_heights'][x],reverse=True);
        #     candidate_pkidx = tmp[1]; # 2nd widest peak
        #     wax_bottom_depth = sliceinfo['pksB'][0][candidate_pkidx];
    else:
        # use pksB exclusively
        print('condition catch: 1 using pksB exclusively');
        pks = sliceinfo['pksB'];
        pksqualifying = np.where(pks[0] > sliceinfo['pixel_depth_strip_bot']-sliceinfo['pixel_depth_strip_top'])[0]; # all to the right of the yellow line
        tmp = sorted(list(range(pks[0][pksqualifying].shape[0])),key=lambda x: pks[1]['width_heights'][pksqualifying][x],reverse=True);
        candidate_pkidx = tmp[0]; # Widest peak past the yellow-line
        wax_bottom_depth = pks[0][pksqualifying][candidate_pkidx];

        #raise ValueError('here');

    if(last_pos is not None):
        pkerrors = pks[0]-last_pos;
        closest_pk_idx=np.argmin(abs(pkerrors))
        print(f'Last = {last_pos} pksBerrorsFromLast = {pkerrors} thePeakIdx = {closest_pk_idx}');
    
        wax_bottom_depth = pks[0][closest_pk_idx].item();
    
        # maximum error?
        max_error_from_last = 35;
        if(all(abs(pkerrors)>max_error_from_last)):
            #raise RuntimeError();
            # use the number, but don't set last_pos
            set_last_pos = False;
        else:
            # set last pos
            #last_pos = candidate_wax_position;
            set_last_pos = True;


    return candidate_pkidx,wax_bottom_depth,set_last_pos;

sliceinfo = dfstepA.iloc[halfway_datalist_index];
#sliceinfo = dfstepA.loc[2495];
candidate_pkidx, initialize_wax_bottom_depth_px, set_last_pos = guess_waxbottom_from_peaks_close_to_lastvalue(sliceinfo);

# if True:
#     # diagnostic plot
#     fig = plt.figure(figsize=(8,4));
#     gs = fig.add_gridspec(4,1);


print( f'Initialize Candidate Peak Index={candidate_pkidx} WaxBottomDepth={initialize_wax_bottom_depth_px}px' );




# --DO RIGHT-HALF OF STRIP--
last_pos = initialize_wax_bottom_depth_px;
wax_bottom_pos_righthalf = [];
for slice_center in slice_centers[halfway_datalist_index:]:
    sliceinfo = dfstepA.loc[slice_center]
    print(f'Slice px={slice_center}');

    candidate_pkidx,candidate_wax_position,set_last_pos = guess_waxbottom_from_peaks_close_to_lastvalue(sliceinfo,last_pos);
    if(set_last_pos):
        last_pos = candidate_wax_position;

    print('Position Of Wax Bottom Is: {:} Last was: {:}'.format(candidate_wax_position,last_pos));
    wax_bottom_pos_righthalf.append( candidate_wax_position );

# --DO LEFT-HALF OF STRIP--
last_pos = initialize_wax_bottom_depth_px;
wax_bottom_pos_lefthalf = [];
for slice_center in slice_centers[halfway_datalist_index:0:-1]:
    sliceinfo = dfstepA.loc[slice_center]
    print(f'Slice px={slice_center}');

    candidate_pkidx,candidate_wax_position,set_last_pos = guess_waxbottom_from_peaks_close_to_lastvalue(sliceinfo,last_pos);
    if(set_last_pos):
        last_pos = candidate_wax_position;

    print('Position Of Wax Bottom Is: {:} Last was: {:}'.format(candidate_wax_position,last_pos));
    wax_bottom_pos_lefthalf.append( candidate_wax_position );

In [ ]:
# --COMBINE TOGETHER--
# Combined left-side and right-side
final_wax_bot = np.concat([np.array(wax_bottom_pos_lefthalf[::-1]),wax_bottom_pos_righthalf])
final_wax_centers = slice_centers;

# make dataframe
dfstepB = pd.DataFrame({'final_wax_bot_subset_px':final_wax_bot},index=slice_centers)
dfstepB.index.name = 'slice'
dfstepB.to_hdf(octstudy.folder_study_processed/'along_strip_data_extracted.hdf5',key='dfstepB')

In [ ]:
# Wax Thickness
wax_top = dfstepA.loc[final_wax_centers]['wax_top_seed_candidate_px'].apply(lambda x: x[0]);
#wax_top = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']
wax_bot = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']+final_wax_bot
wax_thickness = wax_bot-wax_top

px.scatter(x=slice_centers,y=wax_thickness)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

nrows = 1
fig = make_subplots(rows=nrows,shared_xaxes=True)

# Wax Thickness
wax_top = dfstepA.loc[final_wax_centers]['wax_top_seed_candidate_px'].apply(lambda x: x[0]);
#wax_top = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']
wax_bot = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']+final_wax_bot
wax_thickness = wax_bot-wax_top
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_thickness,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_thickness1'
    ),
    row=1,col=1,
)
#fig.update_yaxes(title='Wax Valve Width (pixels)',col=1);


# Wax Thickness
wax_top = dfstepA.loc[final_wax_centers]['wax_top_seed_candidate_px'].apply(lambda x: x[0]);
#wax_top = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']
wax_bot = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']+final_wax_bot
wax_thickness = wax_bot-dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_thickness,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_thickness2'
    ),
    row=1,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_top,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_top'
    ),
    row=1,col=1,
)

# fig.add_trace(
#     go.Scatter(
#         x = dfstepA.index.values,
#         y = dfstepA['pixel_depth_strip_top'],
#         #type = 'heatmap',
#         #colorscale = 'jet'
#         name='Strip Top Depth (px)'
#     ),
#     row=1,col=1,
# )
# #fig.update_yaxes(title='Wax Valve Width (pixels)',col=1);

# fig.add_trace(
#     go.Scatter(
#         x = dfstepA.index.values,
#         y = dfstepA['px_wax_transverse_edges'].apply(lambda x: np.diff(x).item()),
#         #type = 'heatmap',
#         #colorscale = 'jet'
#         name='Wax Valve Width (px)'
#     ),
#     row=1,col=1,
# )
#fig.update_yaxes(title='Wax Valve Width (pixels)',col=1);



fig.update_xaxes(title='distance along strip (px)',row=nrows,col=1);

fig.update_layout(hovermode='x unified',hoversubplots="axis",spikedistance=-1);
#fig = px.imshow(sum_wax_strip_along_length[:,0:]);

#fig.show(renderer='browser')
fig

# Processing Step 5 - Along-Strip Analysis B (no more global thresholding, look for bottom of wax)

In [ ]:
dfstepB.columns

In [ ]:
#SliceDataCrossSection = namedtuple('SliceDataCrossSection','pixel_depth_strip_top_sum_threshold pixel_depth_strip_top pixel_depth_strip_bot pixel_depth_wax_center px_wax_transverse_edges wax_top_seed_candidate_px')
import matplotlib.patches
import scipy.version
def process_a_crosssection_slice_B(slab_thickness_px,slice_position_along_strip_px, mkplot=True):
    slab_thickness = slab_thickness_px;
    localdata = {};

    sliceinfo = dfstepA.loc[slice_position_along_strip_px]

    slab_thickness = 10;
    #slice_position_along_strip_px = slice_center;
    if(slab_thickness>1):
        #seg_subset = seg[:,slice_position_along_strip_px-slab_thickness//2:slice_position_along_strip_px+slab_thickness//2,:]
        img_subset = simgmerged_longcropped[:,slice_position_along_strip_px-slab_thickness//2:slice_position_along_strip_px+slab_thickness//2,:]
    else:
        #seg_subset = seg[:,slice_position_along_strip_px:slice_position_along_strip_px+1,:]
        img_subset = simgmerged_longcropped[:,slice_position_along_strip_px:slice_position_along_strip_px+1,:]

    #slicex = slice(sliceinfo['wax_top_seed_candidate_px'][1]-data_extracted['auto_wax']['max_transverse_wax_edge_width_px']//2 , sliceinfo['wax_top_seed_candidate_px'][1]+data_extracted['auto_wax']['max_transverse_wax_edge_width_px']//2)
    #slicex = slice(sliceinfo['wax_top_seed_candidate_px'][1] , sliceinfo['wax_top_seed_candidate_px'][1]+data_extracted['auto_wax']['max_transverse_wax_edge_width_px'])
    slicex = slice( round(sliceinfo['px_wax_transverse_edges'][0]) , round(sliceinfo['px_wax_transverse_edges'][1]) )
    #slicex = slice( round(sliceinfo['px_wax_transverse_edges'][0])-10 , round(sliceinfo['px_wax_transverse_edges'][1])+10 )

    #slicey = slice(sliceinfo['pixel_depth_strip_top'] , sliceinfo['pixel_depth_strip_top']+3*(sliceinfo['pixel_depth_strip_bot']-sliceinfo['pixel_depth_strip_top'])  )
    slicey = slice(sliceinfo['pixel_depth_strip_top'] , sliceinfo['pixel_depth_strip_top']+4*(sliceinfo['pixel_depth_strip_bot']-sliceinfo['pixel_depth_strip_top'])  )
    #slicey = slice(sliceinfo['pixel_depth_strip_top'] , orignda.shape[2]  )
    #slicey = slice(sliceinfo['wax_top_seed_candidate_px'][0] , sliceinfo['pixel_depth_strip_top']+3*(sliceinfo['pixel_depth_strip_bot']-sliceinfo['wax_top_seed_candidate_px'][0])  )
    print('slicex',slicex);
    print('slicey',slicey);

    sliceywax = slice(sliceinfo['pixel_depth_strip_top'] , slicey.start+dfstepB.loc[slice_position_along_strip_px]['final_wax_bot_subset_px']  )

    #nda = sitk.GetArrayFromImage(seg_subset);
    #print(nda.shape)
    #nda_mean = np.mean(nda,axis=1);
    #nda_mean_sum = np.sum(nda_mean,axis=0);

    orignda = sitk.GetArrayViewFromImage(img_subset);
    orignda_mean = np.mean(orignda,axis=1);




    fig = plt.figure(figsize=(12,10),constrained_layout=True)
    gs = fig.add_gridspec(4,2);

    fig.suptitle(
    """{:s} --> range={:s}dB to uint8
4thC Slice-by-Slice Cross-section Analysis sizes={:s}px
slice_position_along_strip={:.0f}px and slab_thickness={:.0f}px
""".format(octstudy.name,str((oct_scalar_min,oct_scalar_max)),str(simgmerged_longcropped.GetSize()),slice_position_along_strip_px,slab_thickness_px))



    #ax.imshow( skimage.color.label2rgb(label_image==region.label, orignda_mean ) )
    #ax.imshow(orignda_mean[slicex,slicey])
    #ax.imshow(skimage.color.label2rgb(label_image==region.label, orignda_mean[slicex,slicey] ))
    #ax.imshow(skimage.color.label2rgb(label=label_image==region.label,image=orignda_mean,colors=[(255,0,0),(0,0,255)],alpha=0.01, bg_label=0, bg_color=None))

    ax = fig.add_subplot(gs[0,0]);
    ax.imshow(orignda_mean,cmap='gray',aspect='auto',origin='upper')
    rect = matplotlib.patches.Rectangle( (slicey.start, slicex.start), slicey.stop-slicey.start, slicex.stop-slicex.start, linewidth=1, edgecolor='r', facecolor='none')
    ax.add_patch(rect)
    rect2 = matplotlib.patches.Rectangle( (slicey.start, slicex.start), (slicey.start+dfstepB.loc[slice_position_along_strip_px]['final_wax_bot_subset_px'] )-slicey.start, slicex.stop-slicex.start, linewidth=1, edgecolor='yellow', linestyle='dotted',facecolor='none')
    ax.add_patch(rect2)


    ax = fig.add_subplot(gs[0,1]);
    #ax.set_title('Denoised Wax Region'.format(),y=1.0,verticalalignment='bottom')
    orignda_mean_wax = orignda_mean[slicex,sliceywax];
    orignda_mean_wax_denoised = skimage.filters.rank.median(orignda_mean_wax/255.0, skimage.morphology.disk(2))
    ax.imshow(orignda_mean_wax_denoised,cmap='gray',aspect='auto',origin='upper')
    ax.set_title('1.ROI3 to wax bottom, denoised'.format(),y=1.0,verticalalignment='bottom')

    # Global Otsu Thresholding
    ## -- fig3 - Global OTSU Thresholding
    # Applying multi-Otsu threshold for the default value, generating
    # several classes.
    thresholds = skimage.filters.threshold_multiotsu(orignda_mean_wax_denoised,classes=5)
    # Using the threshold values, we generate the three regions.
    regions = np.digitize(orignda_mean_wax_denoised, bins=thresholds)
    #print('5. Multiotsu Thresholds',thresholds);
    ax = fig.add_subplot(gs[1,1])
    ax.set_title('2.Multi-Otsu of #1\n{:s}\n{:s}'.format(str(thresholds),str(thresholds/255.0)),y=1.0,verticalalignment='bottom')
    ax.imshow(regions,cmap='jet',aspect='auto')
    # localdata['testseg_otsu_regions'] = regions;
    # localdata['testseg_otsu_thresholds'] = thresholds;

    # Adaptive Thresholding
    window_size = 55
    thresh_niblack = skimage.filters.threshold_niblack(orignda_mean_wax_denoised, window_size=window_size, k=0.1)
    #thresh_sauvola = skimage.filters.threshold_sauvola(orignda_mean_wax_denoised, window_size=window_size, k=0.1,r=thresholds[0]/2.0)
    #thresh_sauvola = skimage.filters.threshold_sauvola(orignda_mean_wax_denoised, window_size=window_size, k=0.1,r=thresholds[0]/2.0)
    sauvola_k = 0.06;
    sauvola_r = thresholds[0]*0.2;
    thresh_sauvola = skimage.filters.threshold_sauvola(orignda_mean_wax_denoised, window_size=window_size, k=sauvola_k, r=sauvola_r)

    # ax = fig.add_subplot(gs[2,1])
    # ax.set_title('Thresh Niblack w={:} k=0.1'.format(window_size,),y=1.0,verticalalignment='bottom')
    # ax.imshow(orignda_mean_wax_denoised<thresh_niblack,cmap='jet',aspect='auto')

    ax = fig.add_subplot(gs[2,1])
    ax.set_title('3.Thresh Sauvola w={:} k={:}, r={:.2f} of #1'.format(window_size,sauvola_k,sauvola_r),y=1.0,verticalalignment='bottom')
    ax.imshow(orignda_mean_wax_denoised<thresh_sauvola,cmap='jet',aspect='auto')
    ax.axvline(sliceinfo['wax_top_seed_candidate_px'][0] - sliceinfo['pixel_depth_strip_top'],color='cyan',linestyle='dashed')



    # Show Sauvola overlayed
    seg_thresh_sauvola = orignda_mean_wax_denoised<thresh_sauvola;

    ax = fig.add_subplot(gs[2,0])
    #ax.imshow(sliceseg_sliced,alpha=(sliceseg_sliced!=0).astype(float)*0.5,cmap='jet',aspect='auto',extent=(slicey.start,slicey.start+final_wax_bot[count],slicex.stop,slicex.start),origin='upper')
    ax.imshow(orignda_mean,cmap='gray',aspect='auto',origin='upper');
    ax.imshow(seg_thresh_sauvola,alpha=(seg_thresh_sauvola).astype(float)*0.8,cmap='jet',aspect='auto',extent=(sliceywax.start,sliceywax.stop,slicex.stop,slicex.start),origin='upper')
    ax.set_xlim(0, orignda_mean.shape[1])
    ax.set_ylim(orignda_mean.shape[0],0)
    rect2 = matplotlib.patches.Rectangle( (slicey.start, slicex.start), (slicey.start+dfstepB.loc[slice_position_along_strip_px]['final_wax_bot_subset_px'] )-slicey.start, slicex.stop-slicex.start, linewidth=1, edgecolor='yellow', linestyle='dotted',facecolor='none')
    ax.add_patch(rect2)

    #localdata['seg_thresh_sauvola'] = seg_thresh_sauvola;






    # CUT OFF LEFTMOST SEG




    # Show Sauvola overlayed
    # seg_thresh_sauvola = orignda_mean_wax_denoised<thresh_sauvola;
    region_labels_to_keep = [];
    label_image = skimage.measure.label(seg_thresh_sauvola);
    for region in skimage.measure.regionprops(label_image):
        print('Region',region.label)
        print(region.centroid, sliceinfo['wax_top_seed_candidate_px'][0], sliceinfo['pixel_depth_strip_top'])
        if(region.centroid[1] <= sliceinfo['wax_top_seed_candidate_px'][0] - sliceinfo['pixel_depth_strip_top']):
            # ignore
            continue;
        region_labels_to_keep.append(region.label);
    print('RegionsToKeep',region_labels_to_keep)
    seg_thresh_sauvola_cutoff = np.isin(label_image,region_labels_to_keep)

    # Force everything to zero left of the cutoff point
    if(sliceinfo['wax_top_seed_candidate_px'][0] - sliceinfo['pixel_depth_strip_top'] >=0):
        seg_thresh_sauvola_cutoff[ : , 0 : sliceinfo['wax_top_seed_candidate_px'][0] - sliceinfo['pixel_depth_strip_top'] ] = 0;

    ax = fig.add_subplot(gs[3,1])
    ax.set_title('4. Cut-off leftmost of #3'.format(window_size,sauvola_k,sauvola_r),y=1.0,verticalalignment='bottom')
    ax.imshow(seg_thresh_sauvola_cutoff,cmap='jet',aspect='auto')

    ax = fig.add_subplot(gs[3,0])
    #ax.imshow(sliceseg_sliced,alpha=(sliceseg_sliced!=0).astype(float)*0.5,cmap='jet',aspect='auto',extent=(slicey.start,slicey.start+final_wax_bot[count],slicex.stop,slicex.start),origin='upper')
    ax.imshow(orignda_mean,cmap='gray',aspect='auto',origin='upper');
    ax.imshow(seg_thresh_sauvola_cutoff,alpha=(seg_thresh_sauvola_cutoff).astype(float)*0.8,cmap='jet',aspect='auto',extent=(sliceywax.start,sliceywax.stop,slicex.stop,slicex.start),origin='upper')
    ax.set_xlim(0, orignda_mean.shape[1])
    ax.set_ylim(orignda_mean.shape[0],0)
    rect2 = matplotlib.patches.Rectangle( (slicey.start, slicex.start), (slicey.start+dfstepB.loc[slice_position_along_strip_px]['final_wax_bot_subset_px'] )-slicey.start, slicex.stop-slicex.start, linewidth=1, edgecolor='yellow', linestyle='dotted',facecolor='none')
    ax.add_patch(rect2)
    localdata['seg_thresh_sauvola_cutoff'] = seg_thresh_sauvola_cutoff;

    # localdata = {
    #     'wax_image_sum_along_strip_depth':image_sumx
    # }

    if mkplot:
        fig.tight_layout();
    else:
        fig = None;

    return localdata,fig;

#process_a_crosssection_slice_B(10,255,mkplot=True)
#process_a_crosssection_slice_B(10,2005,mkplot=True)
#process_a_crosssection_slice_B(10,345,mkplot=True)
#process_a_crosssection_slice_B(10,65,mkplot=True)
#process_a_crosssection_slice_B(10,2005,mkplot=True)
process_a_crosssection_slice_B(10,1615,mkplot=True)

In [ ]:
# loop and calculate and make figures
slab_thickness = 10;

mkplot=True;
overwrite=True;

if overwrite:
    datalist2 = [];
#np.linspace(start=slab_thickness//2,stop=Seg.GetSize()[1],num)
slice_centers = list(range(slab_thickness//2, seg_sliced.GetSize()[1], slab_thickness//2))
#slice_centers = list(range(slab_thickness//2, seg_sliced.GetSize()[1], slab_thickness));
import os
filebase = r'figureoutput_20250428_stepB';
for count,slicecenter in enumerate(slice_centers):
    print(count,slicecenter)
    output_filename = r'C:\TEMP\{:s}{:04d}.jpg'.format(filebase,count);

    if( (overwrite and os.path.exists(output_filename)) or (not os.path.exists(output_filename))):
        data,fig = process_a_crosssection_slice_B(slab_thickness,slicecenter,mkplot=mkplot);
        
        if mkplot:
            fig.savefig(output_filename);
            print('Wrote',output_filename);
            plt.close(fig);
        
        datalist2.append(data);
    
    #break;

In [ ]:
# make video from image sequence
import ffmpeg
import glob
import os
try:
    (
    ffmpeg
    .input(r'C:\TEMP\{:s}%04d.jpg'.format(filebase), framerate=30) # Assumes images are named frame_1.png, frame_2.png etc.
    .output(r'C:\TEMP\{:s}.mp4'.format(filebase), crf=20, preset='slower', movflags='faststart', pix_fmt='yuv420p')
    .run(capture_stdout=True, capture_stderr=True, overwrite_output=True)
    )
except ffmpeg.Error as e:
    print('stdout:', e.stdout.decode('utf8'))
    print('stderr:', e.stderr.decode('utf8'))
    raise e
# delete the source .jpgs
for filepath in glob.glob(r'C:\TEMP\{:s}*.jpg'.format(filebase)):
    os.unlink(filepath);


In [ ]:
# process results datalist4
dfstepC = pd.DataFrame(datalist2,index=slice_centers);
dfstepC.index.name = 'slice'
dfstepC.to_hdf(octstudy.folder_study_processed/'along_strip_data_extracted.hdf5',key='dfstepC')

In [ ]:
dfstepC.info()

In [ ]:
# process results datalist4
#for slice_center,sliceinfo3,sliceinfo4 in zip(slice_centers,datalist3,datalist4):
regionparamlist = [];
for count,slice_center in enumerate(slice_centers):
    sliceinfo = dfstepA.loc[slice_center]
    sliceinfoB = dfstepB.loc[slice_center]
    sliceinfoC = dfstepC.loc[slice_center]

    print(f'Slice px={slice_center}');
    #print(sliceinfo,sliceinfoB,sliceinfoC)

    #break;
    segslice = sliceinfoC['seg_thresh_sauvola_cutoff'];
    seglabel = skimage.measure.label(segslice);
    total_area_filled = 0;
    total_area = 0;
    total_area_convex = 0;
    for region in skimage.measure.regionprops(seglabel):
        region : skimage.measure._regionprops.RegionProperties;
        # print('Region id:{:} area:{:} area_filled:{:}'.format(
        #     region.label,
        #     region.area,
        #     region.area_filled
        # ))
        total_area += region.area.item();
        total_area_filled += region.area_filled.item();
        total_area_convex += region.area_convex.item();

        #print(region.centroid, sliceinfo['wax_top_seed_candidate_px'][0], sliceinfo['pixel_depth_strip_top'])

    regionparamlist.append({
        'area':total_area,
        'area_filled':total_area_filled,
        'area_convex':total_area_convex,
    })
dfstepD = pd.DataFrame(regionparamlist,index=slice_centers);
dfstepD.index.name = 'slice';
dfstepD


In [ ]:
plt.imshow(region.image)

In [ ]:
plt.imshow(dfstepC.iloc[200]['seg_thresh_sauvola'])

In [ ]:
slice_center=3485;
sliceinfo = dfstepA.loc[slice_center]
sliceinfoB = dfstepB.loc[slice_center]
sliceinfoC = dfstepC.loc[slice_center]

segslice = sliceinfoC['seg_thresh_sauvola_cutoff'];
seglabel = skimage.measure.label(segslice);

fig = plt.figure(figsize=(10,12));
nrows = 5;
ncols = 4;
gs = fig.add_gridspec(nrows=nrows,ncols=ncols);

ax = fig.add_subplot(gs[0,0:4]);
ax.imshow(sliceinfo['testseg_otsu_regions'],aspect='auto')

ax = fig.add_subplot(gs[1,0:2]);
ax.imshow(segslice,cmap='jet',aspect='auto')


regionparamlist = [];
total_area_filled = 0;
total_area = 0;
total_area_convex = 0;
thiscol = 0;
thisrow = 2;
for region in skimage.measure.regionprops(seglabel):
    region : skimage.measure._regionprops.RegionProperties;
    print('Region id:{:} area:{:} area_filled:{:} area_convex:{:}'.format(
        region.label,
        region.area,
        region.area_filled,
        region.area_convex,
    ))
    total_area += region.area.item();
    total_area_filled += region.area_filled.item();
    total_area_convex += region.area_convex.item();

    # thisrow=thisrow + (region.label-1)//nrows;
    thiscol = (region.label-1)%ncols
    #ax = fig.add_subplot(gs[thisrow,thiscol]);
    ax = fig.add_subplot(gs[thisrow,thiscol]);
    ax.imshow(region.image,aspect='auto',cmap='gray')
    ax = fig.add_subplot(gs[thisrow+1,thiscol]);
    ax.imshow(region.image_filled,aspect='auto',cmap='gray')
    ax = fig.add_subplot(gs[thisrow+2,thiscol]);
    ax.imshow(region.image_convex,aspect='auto',cmap='gray')


# regionparamlist.append({
#     'area':total_area,
#     'area_filled':total_area_filled
# })
# pp.pprint(regionparamlist)
fig.tight_layout()


In [ ]:
plt.imshow(region.image)

In [ ]:
sliceinfo

# Analysis Of Outputs

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

nrows = 3
fig = make_subplots(rows=nrows,shared_xaxes=True,vertical_spacing=0.02, )


legendgroup = 'distances';
# Wax Thickness
wax_top = dfstepA.loc[final_wax_centers]['wax_top_seed_candidate_px'].apply(lambda x: x[0]);
#wax_top = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']
wax_bot = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']+final_wax_bot
wax_thickness = wax_bot-wax_top
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_thickness,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_thickness1',
        legendgroup=legendgroup
    ),
    row=1,col=1,
)
#fig.update_yaxes(title='Wax Valve Width (pixels)',col=1);


# Wax Thickness
wax_top = dfstepA.loc[final_wax_centers]['wax_top_seed_candidate_px'].apply(lambda x: x[0]);
#wax_top = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']
wax_bot = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']+final_wax_bot
wax_thickness = wax_bot-dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_thickness,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_thickness2',
        legendgroup=legendgroup
    ),
    row=1,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = dfstepA['px_wax_transverse_edges'].apply(lambda x: np.diff(x)[0]),
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_width_px',
        legendgroup=legendgroup
    ),
    row=1,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_top,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_top',
        legendgroup=legendgroup
    ),
    row=1,col=1,
)




legendgroup = 'areas';
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = dfstepD['area_filled']-dfstepD['area'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_holes',
        legendgroup=legendgroup
    ),
    row=2,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = dfstepD['area'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area',
        legendgroup=legendgroup
    ),
    row=2,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = dfstepD['area_filled'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_filled',
        legendgroup=legendgroup
    ),
    row=2,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = dfstepD['area_convex'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_convex',
        legendgroup=legendgroup
    ),
    row=2,col=1,
)


legendgroup = 'ratios';
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = dfstepD['area']/dfstepD['area_filled'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_to_areafilled_ratio',
        legendgroup=legendgroup
    ),
    row=3,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = dfstepD['area']/dfstepD['area_convex'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_to_areaconvex_ratio',
        legendgroup=legendgroup
    ),
    row=3,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = dfstepD['area_filled']/dfstepD['area_convex'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_areafilled_to_areaconvex_ratio',
        legendgroup=legendgroup
    ),
    row=3,col=1,
)

# setup legends per row
for i, yaxis in enumerate(fig.select_yaxes(col=1), 1):
    legend_name = f"legend{i}"
    fig.update_layout({legend_name: dict(y=yaxis.domain[1], yanchor="top")}, showlegend=True)
    fig.update_traces(row=i, legend=legend_name)

fig.update_traces(xaxis='x3')
fig.update_yaxes(title='pixels',row=1);
fig.update_yaxes(title='pixels^2',row=2);
fig.update_yaxes(title='ratios',row=3);
fig.update_xaxes(title='distance along strip (px)',row=nrows,col=1);
fig.update_layout(hovermode='x unified',hoversubplots="axis",spikedistance=-1);
#fig = px.imshow(sum_wax_strip_along_length[:,0:]);

fig.show(renderer='browser')
#fig

In [ ]:
fig.data

In [ ]:
dfstepA['px_wax_transverse_edges'].apply(lambda x: np.diff(x)[0])

# Processing Step 1 - Use top-down view to draw the centerline of the wax valve

In [ ]:
print(vdvol)

In [ ]:
pl = vedo_plotters.SimonSlicer3DPlotter(
    vdvol,
    #cmaps=("gist_ncar_r", "jet", "Spectral_r", "hot_r", "bone_r"),
    cmaps=("grey","gist_ncar_r"),
    use_slider3d=False,
    show_histo=False,
    show_icon=False,
    bg="black",
    bg2="gray5",
    slice_X=True,
    slice_Y=False,
    slice_Z=False,
    scalar_range=(30.0,50.0), # dB range from OCT scalars
)

bnds = vdvol.bounds();

#vdLine = vedo.Line(p0=(0,0,bnds[5]/2),p1=(0,bnds[3],bnds[5]/2),closed=False,lw=6,c='red');
try:
    step1info
    # variable is defined
    pts = [step1info['wax_valve_manual_nodes_mm'][0], step1info['wax_valve_manual_nodes_mm'][1]];
except NameError:
    # variable was not defined
    pts = [(0, 0, bnds[5]/2) , (0, bnds[3], bnds[5]/2)];
#pl.add(vdLine);
# Add the spline tool using the same points and interact with it
sptool = pl.add_spline_tool(pts, pc='red', lw=3, closed=False);

# # Add a callback to print some info as the lines is changed
# sptool.add_observer(
#     "end of interaction", 
#     lambda o, e: (
#         print(f"Points changed! Pts= {sptool.nodes()}"),
#     )
# )

pl.parallel_projection(True); # orthographic

# # Can now add any other vedo object to the Plotter scene:
pl += vedo.Text2D(
"""STEP 1 - Find wax stripe

Instructions:

#1. select a slice pixel position using the red slider, if the wax valve region is not visible

#2. drag handles of the line to align with ends of wax valve, and centered along the wax valve

#3. when done press ESC
"""
)

pl.show(
    viewup=[0,0,-1],
    interactive=True,axes=4
);
pl.close()

step1info = dict(
    longitudinal_depth_slicer_voxels=pl.xslider.value,
    wax_valve_manual_nodes_mm=sptool.nodes(),
    longitudinal_depth_slicer_bounds_mm=pl.xslice.bounds(),
)

In [ ]:
print('Step 1 Info:')
pp.pprint(step1info);

# Step 2 - Find Other Dimension

In [ ]:
import math
# origin
slice_origin=(
    step1info['longitudinal_depth_slicer_bounds_mm'][0] ,
    (step1info['wax_valve_manual_nodes_mm'][1][1]-step1info['wax_valve_manual_nodes_mm'][0][1])/2 + step1info['wax_valve_manual_nodes_mm'][0][1] ,
    (step1info['wax_valve_manual_nodes_mm'][1][2]-step1info['wax_valve_manual_nodes_mm'][0][2])/2 + step1info['wax_valve_manual_nodes_mm'][0][2] ,
)
slice_origin = tuple([x.item() for x in slice_origin])
slice_origin

In [ ]:
#v1 = []
v1 = np.diff( np.array( step1info['wax_valve_manual_nodes_mm'] ) ,axis=0)[0];
v2 = np.array([0,1,0]);
v1_u = v1/np.linalg.norm(v1);
v2_u = v2/np.linalg.norm(v2);
print(v1_u)
print(v2_u)
#np.linalg.norm(np.array(step1info['wax_valve_manual_nodes_mm']),axis=1).T
angle_to_y_radians = math.acos(np.dot(v1_u,v2_u))
print('Degrees:',math.degrees(angle_to_y_radians))


def rotate_vector(v, axis, theta):
    """
    Rotates a vector v around an axis by an angle theta.

    Parameters:
    v (numpy.ndarray): The vector to rotate.
    axis (numpy.ndarray): The axis to rotate around.
    theta (float): The angle of rotation in radians.

    Returns:
    numpy.ndarray: The rotated vector.
    """
    axis = axis / np.linalg.norm(axis)  # Normalize the axis
    a = np.cos(theta / 2.0)
    b, c, d = -axis * np.sin(theta / 2.0)
    aa, bb, cc, dd = a * a, b * b, c * c, d * d
    bc, ad, ac, ab, bd, cd = b * c, a * d, a * c, a * b, b * d, c * d
    rotation_matrix = np.array([[aa + bb - cc - dd, 2 * (bc + ad), 2 * (bd - ac)],
                                [2 * (bc - ad), aa + cc - bb - dd, 2 * (cd + ab)],
                                [2 * (bd + ac), 2 * (cd - ab), aa + dd - bb - cc]])
    return np.dot(rotation_matrix, v)

slice_normal = rotate_vector( np.array([0,0,1]) , axis=np.array([1,0,0]), theta=angle_to_y_radians)
print('Normal Vector',slice_normal)

In [ ]:
#slice = vol.copy();
#slice_origin=( step1info['longitudinal_depth_slicer_bounds_mm'][0] , (step1info['wax_valve_manual_nodes_mm'][1][1]-step1info['wax_valve_manual_nodes_mm'][0][1])/2 , (step1info['wax_valve_manual_nodes_mm'][1][2]-step1info['wax_valve_manual_nodes_mm'][0][2])/2 )
vdPlane = vedo.Plane(pos=slice_origin,normal=slice_normal,s=(v1[1],6),c='red7',alpha=0.75)

pl = vedo_plotters.SimonSlicer3DPlotter(
    vdvol,
    #cmaps=("gist_ncar_r", "jet", "Spectral_r", "hot_r", "bone_r"),
    cmaps=("grey","gist_ncar_r"),
    use_slider3d=False,
    show_histo=False,
    show_icon=False,
    bg="black",
    bg2="gray5",
    slice_X=True,
    slice_Y=False,
    slice_Z=False,
    scalar_range=(30.0,50.0), # dB range from OCT scalars
)
#pl = vedo.Plotter();

#pl.add(vdPlane);
vdSlice = vdvol.slice_plane(origin=slice_origin,normal=slice_normal,autocrop=True).cmap('jet',vmin=38.0,vmax=60.0);
pl.add(vdSlice);

# # Can now add any other vedo object to the Plotter scene:
pl += vedo.Text2D(
"""STEP 2 - View Slice Down Center Of Wax Valve

Instructions:

#1. view

#2. when done, press ESC
"""
)

pl.show(
    viewup=[-1,0,0],
    interactive=True,axes=4
);
pl.close()

# Step 3 - Set A Crop Box

In [ ]:
slice_origin=(
    step1info['longitudinal_depth_slicer_bounds_mm'][0] ,
    (step1info['wax_valve_manual_nodes_mm'][1][1]-step1info['wax_valve_manual_nodes_mm'][0][1])/2 + step1info['wax_valve_manual_nodes_mm'][0][1] ,
    (step1info['wax_valve_manual_nodes_mm'][1][2]-step1info['wax_valve_manual_nodes_mm'][0][2])/2 + step1info['wax_valve_manual_nodes_mm'][0][2] ,
)
slice_origin = tuple([x.item() for x in slice_origin])
slice_origin

In [ ]:
# show in pyvista previously sliced vedo plot

pl = pv.Plotter(notebook=False);
pl.background_color='gray'

# add data to plotter
pl.add_mesh(pv.wrap(vdSlice.dataset),cmap='jet',clim=[40,55])

# add interactive widgets
def cb_box(evt):
    print('cb_box',evt)

wdgt = pl.add_box_widget(cb_box, rotation_enabled=False,use_planes=True,factor=1.0,color='red',outline_translation=True);
#wdgt = pl.add_plane_widget(cb_box,normal='x',color='red',normal_rotation=False,factor=0.5,origin=[2,0,0]);
#wdgt = pl.add_plane_widget(cb_box,normal='x',color='green',normal_rotation=False,factor=0.5,origin=[1,0,0]);
#wdgt = pl.add_camera_orientation_widget();
#wdgt = pl.add_box_widget()

# formatting
pl.add_box_axes();
pl.show_bounds();

pl.view_isometric();
pl.view_vector([0,0,1]);
#pl.disable();

pl.show();

In [ ]:
vdSlice = vdvol.slice_plane(origin=slice_origin,normal=slice_normal,autocrop=True).cmap('jet',vmin=38.0,vmax=60.0);
#pvSliceLong = pv.wrap(vdSlice.dataset);

slice_inplane_origin = ( (vdvol.bounds()[1]-vdvol.bounds()[0])/2 , (vdvol.bounds()[3]-vdvol.bounds()[2])/2 , (vdvol.bounds()[5]-vdvol.bounds()[4])/2 );
vdSliceLong = vdvol.slice_plane(origin=slice_inplane_origin,normal=(1,0,0),autocrop=True).cmap('jet',vmin=38.0,vmax=60.0);
#pvSliceLong = pv.wrap(vdSlice.dataset);


pl = pv.Plotter(notebook=False);
pl.background_color='gray'

# add data to plotter
pl.add_mesh(pv.wrap(vdSlice.dataset),cmap='jet',clim=[40,55])
pl.add_mesh(pv.wrap(vdSliceLong.dataset),cmap='jet',clim=[40,55])

# add interactive widgets
def cb_box(evt):
    print('cb_box',evt)


bounds = list(mergedvol.bounds)
# y longitudinal
bounds[2] = step1info['wax_valve_manual_nodes_mm'][0][1].item()
bounds[3] = step1info['wax_valve_manual_nodes_mm'][1][1].item()
# z transverse
bounds[4] = step1info['wax_valve_manual_nodes_mm'][0][2].item()
bounds[5] = step1info['wax_valve_manual_nodes_mm'][1][2].item()
bounds
wdgt = pl.add_box_widget(cb_box, bounds=bounds, rotation_enabled=False,use_planes=True,factor=1.0,color='red',outline_translation=True);
wdgt.SetHandleSize(0.01);
#wdgt = pl.add_plane_widget(cb_box,normal='x',color='red',normal_rotation=False,factor=0.5,origin=[2,0,0]);
#wdgt = pl.add_plane_widget(cb_box,normal='x',color='green',normal_rotation=False,factor=0.5,origin=[1,0,0]);
#wdgt = pl.add_camera_orientation_widget();
#wdgt = pl.add_box_widget()

# rotate the box
# the_box = pv.PolyData()
# wdgt.GetPolyData(the_box)
# the_box.rotate_y(45);
# wdgt.Set
vtkTransform = pv._vtk.vtkTransform()
vtkTransform.RotateX(math.degrees(angle_to_y_radians))
print(vtkTransform.GetMatrix())
wdgt.SetTransform(vtkTransform)


# formatting
pl.add_box_axes();
pl.show_bounds();

#pl.view_isometric();

#pl.enable_parallel_projection()
#pl.enable_image_style()

#pl.enable_parallel_projection();
#pl.disable();

def key_press_callback(key):
    if key == 'w':  # Example: Move forward
        pl.camera.zoom(1.1)  # Zoom in
    elif key == 's':  # Example: Move backward
        pl.camera.zoom(0.9)  # Zoom out
    elif key == 'a':  # Example: Pan left
        #pl.camera.azimuth = pl.camera.azimuth - 10
        pt = list(pl.camera.focal_point);
        pt[1]-=1;
        pl.camera.focal_point = pt;
    elif key == 'd':  # Example: Pan right
        #pl.camera.azimuth = pl.camera.azimuth + 10
        pt = list(pl.camera.focal_point);
        pt[1]+=1;
        pl.camera.focal_point = pt;
    pl.render() # Render the scene after camera changes
# #pl.add_callback('KeyPressEvent', key_press_callback)
#pl.add_key_event(key='a',callback=lambda *_: pl.camera.azimuth += -10)    # pan left
#pl.add_key_event(key='d',callback=lambda *_: pl.camera.azimuth += 10)    # pan right
pl.add_key_event(key='w',callback= lambda: key_press_callback("w") );
pl.add_key_event(key='s',callback= lambda: key_press_callback("s") );
pl.add_key_event(key='a',callback= lambda: key_press_callback("a") );
pl.add_key_event(key='d',callback= lambda: key_press_callback("d") );
#pl.ren_win.AddObserver('KeyPressEvent', key_press_callback);

pl.view_vector([0,0,1]);
#pl.view_yx();
#pl.camera.roll = 90
#pl.camera.azimuth = 180

pl.show();


In [ ]:
the_box = pv.PolyData()
wdgt.GetPolyData(the_box)
print(the_box)
the_box.bounds

In [ ]:
vtkTransform = pv._vtk.vtkTransform()
wdgt.GetTransform(vtkTransform)
print(vtkTransform.GetMatrix())
mtx = vtkTransform.GetMatrix()
dir(mtx)
#mtx.GetData()
vedo.vtk2numpy(vtkTransform)

In [ ]:
step3info = dict(
    box_bounds=the_box.bounds,
    box_transform_4x4=vedo.vtk2numpy(vtkTransform)
)
pp.pprint(step3info)

In [ ]:
bounds

# Step 4. Cross-section Slice Viewer

In [ ]:
print(vdvol)

In [ ]:
#pl = vedo.applications.Slicer2DPlotter(vdvol,levels=(20,60));
#pl = vedo_plotters.SimonSlicer2DPlotter(vdvol,levels=[20,60])
import vedo
vdvol = vedo.Volume(mergedvol);
pl = vedo_plotters.SimonSlicer2DPlotter(
    vdvol,
    histo_color=None,
);
pl.show(
    #viewup=[-1,0,0],
    viewup=[0,1,0],
    interactive=True,
    #axes=4
);
pl.close()


In [ ]:
pl.close()

# Step 5. Test SimpleITK Segmentation Method

In [ ]:
vdvol.dataset.GetPointData().GetScalars().GetNumberOfComponents()

In [ ]:
# Get ITK image without requiring new memory
import itk
import vtk

# create itk_image
itk_image = itk.image_from_vtk_image(vdvol.dataset)

# itk_image
print(itk_image)

In [ ]:
itk_image.GetImageDimension()

In [ ]:
# make simpleitk image
def itkToSimpleITK(itk_image):
    new_sitk_image = sitk.GetImageFromArray(itk.GetArrayViewFromImage(itk_image),isVector=itk_image.GetNumberOfComponentsPerPixel()>1);
    new_sitk_image.SetOrigin(tuple(itk_image.GetOrigin()))
    new_sitk_image.SetSpacing(tuple(itk_image.GetSpacing()))
    new_sitk_image.SetDirection(itk.GetArrayFromMatrix(itk_image.GetDirection()).flatten()) 
    return new_sitk_image;
simgmerged = itkToSimpleITK(itk_image);

In [ ]:
# -- INITIATE SIMPLE ITK UTILITIES FROM THE NOTEBOOKS CODE
import sys
#sys.path.append('sandbox_sg/SimpleITK-Notebooks/Utilities')
sys.path.append('sandbox_sg/SimpleITK-Notebooks/Python')

#from downloaddata import fetch_data as fdata
from myshow import myshow, myshow3d

In [ ]:
# pyvista/vtk to simpleitk
# from SimpleITK.utilities.vtk import vtk2sitk, sitk2vtk
# simgmerged = vtk2sitk(mergedvol);

In [ ]:
print(simgmerged)

In [ ]:
# To visualize the labels image in RGB with needs a image with 0-255 range
sitk_filter_calc_minmax = sitk.MinimumMaximumImageFilter();

sitk_filter_calc_minmax.Execute(simgmerged);
print('Scalar Min:{:} Max:{:}'.format(
      sitk_filter_calc_minmax.GetMinimum(),
      sitk_filter_calc_minmax.GetMaximum()
));

# rescale d image to a narrower fixed range, normalize to 0-255, and cast to an unsigned integer
sitk_filter = sitk.IntensityWindowingImageFilter();
sitk_filter.SetOutputMaximum(255);
sitk_filter.SetOutputMinimum(0);
sitk_filter.SetWindowMinimum(30);
sitk_filter.SetWindowMaximum(60);
#simg_processed = sitk_filter.Execute(simgmerged);
simg_processed = sitk.Cast(sitk_filter.Execute(simgmerged), sitk.sitkUInt8)
# #simg_processed = sitk
# #img_T1_255 = sitk.Square(simgmerged);

sitk_filter_calc_minmax.Execute(simg_processed);
print('Scalar Min:{:} Max:{:}'.format(
      sitk_filter_calc_minmax.GetMinimum(),
      sitk_filter_calc_minmax.GetMaximum()
));

In [ ]:
print(simg_processed)

In [ ]:
myshow3d(simg_processed)


In [ ]:
del simgmerged
del sitk_filter
del sitk_filter_calc_minmax


In [ ]:
del octstudy

In [ ]:
del mergedvol

## Tresholding

### Basic Threshold

In [ ]:
seg = simg_processed > 200
myshow(sitk.LabelOverlay(simg_processed, seg), "Basic Thresholding")
#sitk.Show(sitk.LabelOverlay(simg_processed, seg))

### Binary Thresholding

In [ ]:
seg = sitk.BinaryThreshold(
    simgmerged, lowerThreshold=250, upperThreshold=255, insideValue=1, outsideValue=0
)
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow( sitk.LabelOverlay(simgmerged, seg), title="Binary Thresholding", dpi=10, defaultslice=110 )

### Thresholding: Otsu

In [ ]:
otsu_filter = sitk.OtsuThresholdImageFilter()
otsu_filter.SetInsideValue(0)
otsu_filter.SetOutsideValue(1)
seg = otsu_filter.Execute(simgmerged)
title="Otsu Thresholding [Threshold={:}]".format(otsu_filter.GetThreshold());

from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow( sitk.LabelOverlay(simgmerged, seg), title=title, dpi=10, defaultslice=110 )
# print(otsu_filter.GetThreshold())
# print(otsu_filter.GetNumberOfHistogramBins());

In [ ]:
del otsu_filter

### Thresholding: Binary Thresholding On Entire Dataset (finding bounds)

### Thresholding: Binary Thresholding On A Slice In The Middle

In [ ]:
size = simgmerged.GetSize()
slice_x = slice(0,size[1])
slice_y = slice(size[1]//2-(size[1]//4) , size[1]//2+(size[1]//4) )
simgmerged_midstrip = simgmerged[:,slice_y,:]
print(simgmerged_midstrip.GetSize())
#from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
#sitk_myshow(simgmerged_midstrip)
#sitk_plotters.ImageSITKSliceViewer3DJupyter(simgmerged_midstrip)

In [ ]:
seg = sitk.BinaryThreshold(
    simgmerged_midstrip, lowerThreshold=250, upperThreshold=255, insideValue=1, outsideValue=0
)
#from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
#sitk_myshow( sitk.LabelOverlay(simgmerged_midstrip, seg), title="Binary Thresholding", dpi=10, defaultslice=110 )

In [ ]:
#sitk_plotters.ImageSITKSliceViewer3DJupyter(  sitk.LabelOverlay(simgmerged_midstrip, seg) )
#sitk_plotters.ImageSITKSliceViewer3DJupyter(  simgmerged_midstrip )
simg_overlay = sitk.LabelOverlay(simgmerged_midstrip, seg);
sitk_plotters.ImageSITKSliceViewer3DJupyter(  simg_overlay )

In [ ]:
# find a probable seed-point in the middle of the test strip
slab_thickness = 10;
seg_subset = seg[:,seg.GetSize()[1]//2-slab_thickness//2:seg.GetSize()[1]//2+slab_thickness//2,:]
sitk_plotters.ImageSITKSliceViewer3DJupyter( seg_subset )

In [ ]:
nda = sitk.GetArrayFromImage(seg_subset)

In [ ]:
octstudy.name

In [ ]:
SliceDataCrossSection = namedtuple('SliceDataCrossSection','pixel_depth_strip_top_sum_threshold pixel_depth_strip_top pixel_depth_strip_bot pixel_depth_wax_center px_wax_transverse_edges wax_top_seed_candidate_px')
def process_a_crosssection_slice(seg : sitk.Image, slab_thickness_px,slice_position_along_strip_px, mkplot=True):
    slab_thickness = slab_thickness_px;
    
    if(slab_thickness>1):
        #seg_subset = seg[:,seg.GetSize()[1]//2-slab_thickness//2:seg.GetSize()[1]//2+slab_thickness//2,:]
        seg_subset = seg[:,slice_position_along_strip_px-slab_thickness//2:slice_position_along_strip_px+slab_thickness//2,:]
    else:
        seg_subset = seg[:,slice_position_along_strip_px:slice_position_along_strip_px+1,:]

    nda = sitk.GetArrayFromImage(seg_subset);

    nda_mean = np.mean(nda,axis=1);
    nda_mean_sum = np.sum(nda_mean,axis=0);
    #np.nonzero(nda_mean_sum>5)[0];

    if mkplot:
        fig,ax = plt.subplots(3,2,figsize=(12,8));
        ax[0,1].imshow(nda_mean,aspect='auto',cmap='grey');
        ax[1,1].plot(nda_mean_sum);
    if mkplot:
        fig.suptitle('{:s} --> range={:s}dB to uint8\nAnalysis of threshold_segmentation={:s} of with size={:s}px\nslice_position_along_strip={:.0f}px and slab_thickness={:.0f}px'.format(octstudy.name,str((oct_scalar_min,oct_scalar_max)),str((250,255)),str(seg.GetSize()),slice_position_along_strip_px,slab_thickness_px))


    pixel_depth_strip_top_sum_threshold = 100;
    pixel_depth_strip_top = np.nonzero(nda_mean_sum>pixel_depth_strip_top_sum_threshold)[0][0];
    pixel_depth_strip_bot = np.nonzero(nda_mean_sum>pixel_depth_strip_top_sum_threshold)[0][-1];

    if mkplot:
        ax[1,1].set_title('Depth-Mean And X-Sum Vs Y',y=0.95,verticalalignment='top')
        ax[1,1].plot(pixel_depth_strip_top, nda_mean_sum[pixel_depth_strip_top] , marker='^', markersize=10,color='orange');
        ax[0,1].axvline(pixel_depth_strip_top,linestyle='--',color='orange');
        ax[1,1].axvline(pixel_depth_strip_top,linestyle='--',color='orange');
        ax[1,1].plot(pixel_depth_strip_bot, nda_mean_sum[pixel_depth_strip_bot] , marker='^', markersize=10,color='yellow');
        ax[1,1].axvline(pixel_depth_strip_bot,linestyle='--',color='yellow');
        ax[0,1].axvline(pixel_depth_strip_bot,linestyle='--',color='yellow');


    if mkplot:
        #ax[2,1].plot(np.diff(nda_mean_sum,1));
        ax[2,1].set_title('Savgol 2nd Derivative Of Above',y=0.95,verticalalignment='top')
        ax[2,1].plot(scipy.signal.savgol_filter(nda_mean_sum,21,polyorder=3,deriv=2));

    if mkplot:
        ax[0,0].set_title('Inverted SubROI',y=0.95,verticalalignment='bottom')
    imgtmp = nda_mean[:,pixel_depth_strip_top:pixel_depth_strip_bot];


    if mkplot:
        ax[0,0].imshow(np.max(imgtmp)-imgtmp);


    pixel_depth_wax_center = round( (pixel_depth_strip_bot-pixel_depth_strip_top)/2 + pixel_depth_strip_top )
    if mkplot:
        ax[0,1].axvline(pixel_depth_wax_center,linestyle='--',color='white');
        ax[0,0].axvline(pixel_depth_wax_center-pixel_depth_strip_top,linestyle='--',color='white');
    
    if mkplot:
        ax[1,0].set_title('Inverted Sum Of Depth along WhiteLine',y=0.95,verticalalignment='top');

    sum_of_depth_intensities = np.sum(imgtmp,axis=1);
    sum_of_depth_intensities = max(sum_of_depth_intensities) - sum_of_depth_intensities;
    if mkplot:
        ax[1,0].plot(sum_of_depth_intensities)

    #pks = scipy.signal.find_peaks(sum_of_depth_intensities,width=20,prominence=1.2)
    pks = scipy.signal.find_peaks(sum_of_depth_intensities,height=25,prominence=25)
    widths = scipy.signal.peak_widths(sum_of_depth_intensities,pks[0],rel_height=0.5)
    print(pks)
    print(widths)
    
    if mkplot:
        ax[1,0].plot(pks[0],sum_of_depth_intensities[pks[0]],linestyle='none',marker='.',color='blue')
    assert(pks[0].shape[0]>=1)
    #pkcenter = (pks[1]['right_edges'][1] - pks[1]['left_edges'][0])//2 + pks[1]['left_edges'][0];
    #pkcenter = int(round( widths[3][0] - widths[2][0] )//2 + widths[2][0]);
    #pkcenter = int(round( max(widths[3])- min(widths[2]) )//2 + min(widths[2]));
    # retain widths of the single largest peak
    pkidxlargest = np.argmax(widths[0]);
    pkcenter = int(round( widths[3][pkidxlargest]- widths[2][pkidxlargest] )//2 + widths[2][pkidxlargest]);
    #ax[1,0].plot(pkcenter,sum_of_depth_intensities[pkcenter],linestyle='none',marker='.',color='red')
    if mkplot:
        ax[0,1].axhline(pkcenter,linestyle='--',color='red');
        ax[0,0].axhline(pkcenter,linestyle='--',color='red');
        ax[0,0].axhline(pkcenter,linestyle='--',color='red');
    #imgtmp = nda_mean[:,pixel_depth_strip_top:pixel_depth_strip_bot];
    #pks2d = skimage.feature.peak_local_max(-imgtmp,min_distance=20,num_peaks=2)
    #px_wax_transverse_edges = ( min(pks[1]['left_edges']), max(pks[1]['right_edges']) )
    #px_wax_transverse_edges = ( widths[2][0].item() , widths[3][0].item() )
    #px_wax_transverse_edges = ( min(widths[2]).item() , max(widths[3]).item() )
    px_wax_transverse_edges = ( widths[2][pkidxlargest].item() , widths[3][pkidxlargest].item() )
    if mkplot:
        ax[0,1].axhline(px_wax_transverse_edges[0],linestyle='--',color='purple');
        ax[0,1].axhline(px_wax_transverse_edges[1],linestyle='--',color='purple');
        ax[0,0].axhline(px_wax_transverse_edges[0],linestyle='--',color='purple');
        ax[0,0].axhline(px_wax_transverse_edges[1],linestyle='--',color='purple');
        ax[1,0].axvline(px_wax_transverse_edges[0],linestyle='--',color='purple');
        ax[1,0].axvline(px_wax_transverse_edges[1],linestyle='--',color='purple');


    # take cross-section along white dashed line
    # take intensity over dash red line out of plane of strip
    if mkplot:
        ax[2,0].set_title('Intensity Red Dashed',y=0.95,verticalalignment='top');
    red_outofplane_intensity = nda_mean[pkcenter,:];
    if mkplot:
        ax[2,0].plot(red_outofplane_intensity)
    #ax[2,0].plot(  )

    # specify a seed point
    #pks = scipy.signal.find_peaks(red_outofplane_intensity,distance=30)
    #wax_top_seed_candidate_px = (pks[0][0].item(),pkcenter)
    pks = scipy.signal.find_peaks(red_outofplane_intensity[0:pixel_depth_strip_bot],height=0.59,prominence=0)
    widths = scipy.signal.peak_widths(red_outofplane_intensity[0:pixel_depth_strip_bot],pks[0],rel_height=0.5)
    pkidxlargest = np.argmax(widths[0]);
    wax_top_seed_candidate_px = (round((pks[1]['right_bases'][pkidxlargest]-pks[1]['left_bases'][pkidxlargest])/2 + pks[1]['left_bases'][pkidxlargest]) ,pkcenter)
    print('pks2',pks)
    print('wid2',widths)
    if mkplot:
        ax[2,0].plot(pks[0],red_outofplane_intensity[pks[0]],linestyle='none',marker='.',color='blue')
        ax[0,1].plot(*wax_top_seed_candidate_px, 'x', markersize=10 , color='cyan' );
        ax[2,0].axvline(wax_top_seed_candidate_px[0],linestyle='--',color='cyan');

    # slice_data_cross_section = dict(
    #     pixel_depth_strip_top_sum_threshold=pixel_depth_strip_top_sum_threshold,
    #     pixel_depth_strip_top=pixel_depth_strip_top.item(),
    #     pixel_depth_strip_bot=pixel_depth_strip_bot.item(),
    #     pixel_depth_wax_center=pixel_depth_wax_center,
    #     px_wax_transverse_edges=px_wax_transverse_edges
    # )
    # return slice_data_cross_section;
    return (
        SliceDataCrossSection(
            pixel_depth_strip_top_sum_threshold, pixel_depth_strip_top.item(), pixel_depth_strip_bot.item(), pixel_depth_wax_center, px_wax_transverse_edges,
            wax_top_seed_candidate_px
        ),
        fig
    )

#process_a_crosssection_slice(seg,10,seg.GetSize()[1]//2)
#process_a_crosssection_slice(seg,10,450)
#process_a_crosssection_slice(seg,10,2760)
#process_a_crosssection_slice(seg,10,595)
#process_a_crosssection_slice(seg,10,335)
#process_a_crosssection_slice(seg,10,2005)
#process_a_crosssection_slice(seg,10,1035)
process_a_crosssection_slice(seg,10,1480)

In [ ]:
# loop and calculate and make figures
slab_thickness = 10;
seg.GetSize()[1]
datalist = [];
#np.linspace(start=slab_thickness//2,stop=Seg.GetSize()[1],num)
slice_centers = list(range(slab_thickness//2, seg.GetSize()[1], slab_thickness//2))
for count,slicecenter in enumerate(slice_centers):
    print(count,slicecenter)
    output_filename = r'C:\TEMP\figureoutput{:04d}.jpg'.format(count);
    data,fig = process_a_crosssection_slice(seg,slab_thickness,slicecenter);
    fig.savefig(output_filename);
    print('Wrote',output_filename);
    plt.close(fig);
    
    datalist.append(data);

In [ ]:
# make video from image sequence
import ffmpeg
import glob
import os
try:
    (
    ffmpeg
    .input(r'C:\TEMP\figureoutput%04d.jpg', framerate=30) # Assumes images are named frame_1.png, frame_2.png etc.
    .output(r'C:\TEMP\figureoutput.mp4', crf=20, preset='slower', movflags='faststart', pix_fmt='yuv420p')
    .run(capture_stdout=True, capture_stderr=True, overwrite_output=True)
    )
except ffmpeg.Error as e:
    print('stdout:', e.stdout.decode('utf8'))
    print('stderr:', e.stderr.decode('utf8'))
    raise e
# delete the source .jpgs
for filepath in glob.glob(r'C:\TEMP\figureoutput*.jpg'):
    os.unlink(filepath);


In [ ]:
dfstepA = pd.DataFrame.from_records(datalist,columns=datalist[0]._fields);
dfstepA['slice'] = slice_centers

In [ ]:
dfstepA

In [ ]:
seg_subset.GetSize()

In [ ]:
dfstepA

In [ ]:
seg.GetSize()[1]

In [ ]:
round(pixel_depth_wax_center)

In [ ]:
plt.imshow(nda_mean[:,pixel_depth_strip_top:pixel_depth_strip_bot])

In [ ]:
nda_mean_sum.shape

In [ ]:
scipy.signal.find_peaks(abs(scipy.signal.savgol_filter(nda_mean_sum,21,polyorder=3,deriv=2)),distance=5,width=5)

In [ ]:

print('Top Of Strip At {:d}px depth'.format(pixel_depth_strip_top))
#pixel_depth_strip_top

### Thresholding: Otsu On A Slice In The Middle

In [ ]:
size = simgmerged.GetSize()
slice_x = slice(0,size[1])
slice_y = slice(size[1]//2-(size[1]//4) , size[1]//2+(size[1]//4) )
simgmerged_midstrip = simgmerged[:,slice_y,:]
print(simgmerged_midstrip.GetSize())
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow(simgmerged_midstrip)

In [ ]:
size = simgmerged.GetSize()
slice_x = slice(0,size[1])
slice_y = slice(size[1]//2-(size[1]//4) , size[1]//2+(size[1]//4) )
print(slice_y)
simgmerged_midstrip = simgmerged[:,slice_y,:]
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
simgmerged_midstrip_array = sitk.GetArrayViewFromImage(simgmerged_midstrip)

In [ ]:
np.sum(np.sum(simgmerged_midstrip_array,axis=0),axis=1).shape

In [ ]:
otsu_filter = sitk.OtsuThresholdImageFilter()
otsu_filter.SetInsideValue(0)
otsu_filter.SetOutsideValue(1)
otsu_filter.SetNumberOfHistogramBins(8)
#otsu_filter.
seg = otsu_filter.Execute(simgmerged_midstrip)
title="Otsu Thresholding [Threshold={:}]".format(otsu_filter.GetThreshold());

from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow( sitk.LabelOverlay(simgmerged_midstrip, seg), title=title, dpi=10, defaultslice=110 )
print(otsu_filter.GetThreshold())
print(otsu_filter.GetNumberOfHistogramBins());

### Thresholding: Maximum Entropy

In [ ]:
filter_threshold_maxentropy = sitk.MaximumEntropyThresholdImageFilter();
filter_threshold_maxentropy.SetInsideValue(0)
filter_threshold_maxentropy.SetOutsideValue(1)
seg = filter_threshold_maxentropy.Execute(simg_processed)
title="MaximumEntropyThresholdImageFilter [Threshold={:}]".format(filter_threshold_maxentropy.GetThreshold());
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow( sitk.LabelOverlay(simg_processed, seg), title=title, dpi=10, defaultslice=110 )
print(filter_threshold_maxentropy.GetThreshold())
print(filter_threshold_maxentropy.GetNumberOfHistogramBins());

In [ ]:
del filter_threshold_maxentropy

### Thresholding: Huang

In [ ]:
filter_threshold_huang = sitk.HuangThresholdImageFilter();
filter_threshold_huang.SetInsideValue(0)
filter_threshold_huang.SetOutsideValue(1)
seg = filter_threshold_huang.Execute(simg_processed)
title="HuangThresholdImageFilter [Threshold={:}]".format(filter_threshold_huang.GetThreshold());
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow( sitk.LabelOverlay(simg_processed, seg), title=title, dpi=10, defaultslice=110 )
print(filter_threshold_huang.GetThreshold())
print(filter_threshold_huang.GetNumberOfHistogramBins());

In [ ]:
del filter_threshold_huang

In [ ]:
plt.close('all')

## Region Growing Segmentation

The first step of improvement upon the naive thresholding is a class of algorithms called region growing. This includes:
<ul>
  <li><a href="http://www.itk.org/Doxygen/html/classitk_1_1ConnectedThresholdImageFilter.html">ConnectedThreshold</a></li>
  <li><a href="http://www.itk.org/Doxygen/html/classitk_1_1ConfidenceConnectedImageFilter.html">ConfidenceConnected</a></li>
  <li><a href="http://www.itk.org/Doxygen/html/classitk_1_1VectorConfidenceConnectedImageFilter.html">VectorConfidenceConnected</a></li>
  <li><a href="http://www.itk.org/Doxygen/html/classitk_1_1NeighborhoodConnectedImageFilter.html">NeighborhoodConnected</a></li>
</ul>

Earlier we used 3D Slicer to determine that index: (132,142,96) was a good seed for the left lateral ventricle.

In [ ]:
sitk.Show(simg_processed)

In [ ]:
#seed = (132, 142, 96)
seed = (222,5247,106)
seg = sitk.Image(simgmerged.GetSize(), sitk.sitkUInt8)
seg.CopyInformation(simgmerged)
seg[seed] = 1
seg = sitk.BinaryDilate(seg, [3] * 3)
#sitk_myshow(sitk.LabelOverlay(simg_processed, seg), "Initial Seed");

### Connected Threshold

In [ ]:
seed = (222,5247,106); # found using fiji, coordinate on top surface of the device at a particular point
seg = sitk.ConnectedThreshold(simgmerged, seedList=[seed], lower=225, upper=255);
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow(sitk.LabelOverlay(simgmerged, seg), "Connected Threshold");
#sitk.Show( sitk.LabelOverlay(simg_processed, seg) );

### Confidence-Connected-Threshold (automatically set a threshold based on neighborhood)

In [ ]:
# Improving upon this is the ConfidenceConnected filter, which uses the initial seed or current segmentation to estimate the threshold range.
seed = (222,5247,106); # found using fiji, coordinate on top surface of the device at a particular point
seg = sitk.ConfidenceConnected(
    simgmerged,
    seedList=[seed],
    numberOfIterations=1,
    multiplier=1.0,
    initialNeighborhoodRadius=1,
    replaceValue=1,
)
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow(sitk.LabelOverlay(simgmerged, seg), "ConfidenceConnected");
#sitk.Show(sitk.LabelOverlay(simgmerged, seg))

In [ ]:
# query this segmentation
filter_label_statistics = sitk.LabelStatisticsImageFilter();
filter_label_statistics.Execute(simgmerged,seg);


In [ ]:

# Get the label values that have statistics
labels = filter_label_statistics.GetLabels()
print(f"Labels: {labels}")

# Iterate over each label and print statistics
for label in labels:
    print(f"Statistics for label: {label}");
    print(f"  Mean: {filter_label_statistics.GetMean(label)}");
    print(f"  Minimum: {filter_label_statistics.GetMinimum(label)}");
    print(f"  Maximum: {filter_label_statistics.GetMaximum(label)}");
    print(f"  Sum: {filter_label_statistics.GetSum(label)}");
    print(f"  Variance: {filter_label_statistics.GetVariance(label)}");
    print(f"  Standard Deviation: {filter_label_statistics.GetSigma(label)}");
    print(f"  BoundingBox: {str(filter_label_statistics.GetBoundingBox(label))}");
    print(f"  Region: {str(filter_label_statistics.GetRegion(label))}");
    #print(f"  Number of pixels: {filter_label_statistics.GetNumberOfPixels(label)}")

# Test
#filter_label_statistics.

## Fast Marching Segmentation

The FastMarchingImageFilter implements a fast marching solution to a simple level set evolution problem (eikonal equation). In this example, the speed term used in the differential equation is provided in the form of an image. The speed image is based on the gradient magnitude and mapped with the bounded reciprocal $1/(1+x)$.


In [ ]:
#seed = (132, 142, 96)
seed = (222,5247,106); # found using fiji, coordinate on top surface of the device at a particular point
feature_img = sitk.GradientMagnitudeRecursiveGaussian(simg_processed, sigma=0.5)
speed_img = sitk.BoundedReciprocal(
    feature_img
)  # This is parameter free unlike the Sigmoid
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow(speed_img)

In [ ]:
del feature_img
del speed_img

## Level-Set Segmentation

There are a variety of level-set based segmentation filter available in ITK:
<ul>
<li><a href="http://www.itk.org/Doxygen/html/classitk_1_1GeodesicActiveContourLevelSetImageFilter.html">GeodesicActiveContour</a></li>
<li><a href="http://www.itk.org/Doxygen/html/classitk_1_1ShapeDetectionLevelSetImageFilter.html">ShapeDetection</a></li>
<li><a href="http://www.itk.org/Doxygen/html/classitk_1_1ThresholdSegmentationLevelSetImageFilter.html">ThresholdSegmentation</a></li>
<li><a href="http://www.itk.org/Doxygen/html/classitk_1_1LaplacianSegmentationLevelSetImageFilter.html">LaplacianSegmentation</a></li>
<li><a href="http://www.itk.org/Doxygen/html/classitk_1_1ScalarChanAndVeseDenseLevelSetImageFilter.html">ScalarChanAndVese</a></li>
</ul>

There is also a <a href="http://www.itk.org/Doxygen/html/group__ITKLevelSetsv4.html">modular Level-set framework</a> which allows composition of terms and easy extension in C++.




First we create a label image from our seed.

In [ ]:
seed = (222,5247,106); # found using fiji, coordinate on top surface of the device at a particular point

seg = sitk.Image(simg_processed.GetSize(), sitk.sitkUInt8)
seg.CopyInformation(simg_processed)
seg[seed] = 1
seg = sitk.BinaryDilate(seg, [3] * 3)

Use the seed to estimate a reasonable threshold range.

In [ ]:
stats = sitk.LabelStatisticsImageFilter();
stats.Execute(simg_processed, seg);

factor = 1.0;
lower_threshold = stats.GetMean(1) - factor * stats.GetSigma(1);
#upper_threshold = stats.GetMean(1) + factor * stats.GetSigma(1)
lower_threshold = 100;
upper_threshold = 255;
print(lower_threshold, upper_threshold);

In [ ]:
init_ls = sitk.SignedMaurerDistanceMap(seg, insideIsPositive=True, useImageSpacing=True)

In [ ]:
lsFilter = sitk.ThresholdSegmentationLevelSetImageFilter()
lsFilter.SetLowerThreshold(lower_threshold)
lsFilter.SetUpperThreshold(upper_threshold)
lsFilter.SetMaximumRMSError(0.0002)
lsFilter.SetNumberOfIterations(2)
lsFilter.SetCurvatureScaling(2.0)
lsFilter.SetPropagationScaling(1)
lsFilter.ReverseExpansionDirectionOn()
ls = lsFilter.Execute(init_ls, sitk.Cast(simg_processed, sitk.sitkFloat32))
print(lsFilter)

In [ ]:
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow(sitk.LabelOverlay(simg_processed, ls > 0))

## Gradient Watersheds Segmentation

In [ ]:
sigma = simgmerged.GetSpacing()[0]
level = 4

In [ ]:
feature_img = sitk.GradientMagnitude(simgmerged);
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow(feature_img);

In [ ]:
ws_img = sitk.MorphologicalWatershed(
    feature_img, level=0, markWatershedLine=True, fullyConnected=False
)
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow(sitk.LabelToRGB(ws_img), "Watershed Over Segmentation")

# Test Line Extraction

In [ ]:
#pvvol = octstudy.octdatalist[10].pvvol;
pvvol = mergedvol;
pl = pv.Plotter(notebook=False);


#plt.add_volume(pvvol,opacity=0.1);
#pl.add_volume_clip_plane(pvvol,normal='x',assign_to_axis='x',cmap='grey',clim=[40,55]);
#pl.add_mesh_slice(pvvol,normal='x',assign_to_axis='x',cmap='grey',clim=[40,55]);
#pl.add_mesh_slice(pvvol,normal='y',assign_to_axis='y',cmap='grey',clim=[40,55]);
#pl.add_mesh_slice(pvvol,normal='y',cmap='grey',clim=[40,55]);
pl.add_mesh_slice(pvvol,normal='z',assign_to_axis='z',cmap='grey',clim=[40,55]);
#pl.add_mesh_slice(pvvol,normal='z',cmap='grey',clim=[40,55]);
#pl.add_mesh_slice_orthogonal(pvvol,cmap='grey',clim=[40,55]);

#pl.add_volume(pvvol,opacity=0.2);
# def callback(normal, origin):
#     slc = pvvol.slice(normal=normal, origin=origin)
#     origin = list(origin)
#     origin[2] = slc.bounds[5]
#     peak_plane = pv.Plane(
#         center=origin,
#         direction=[0, 0, 1],
#         i_size=20,
#         j_size=20,
#     )
#     _ = pl.add_mesh(
#         peak_plane, name="Peak", color='red', opacity=0.4
#     )
# _ = pl.add_plane_widget(callback, normal_rotation=False)

#pl.add_axes();

# def move_center(pointa, pointb):
#     center = (np.array(pointa) + np.array(pointb)) / 2
#     normal = np.array(pointa) - np.array(pointb)
#     single_slc = pvvol.slice(normal=normal, origin=center)

#     _ = pl.add_mesh(single_slc, name="slc")

# _ = pl.add_line_widget(callback=move_center, use_vertices=True)
pl.add_box_axes();
pl.show_bounds();

pl.show();

In [ ]:
p.plane_sliced_meshes[0].plot(notebook=False)

In [ ]:
from SimpleITK.utilities.vtk import vtk2sitk
simgmerged = vtk2sitk(mergedvol)

In [ ]:
plt = pv.Plotter(notebook=False);
plt.add_volume(mergedvol);
plt.show(jupyter_backend='none')


In [ ]:
import cv2